In [1]:
import os
os.chdir('/workspace/175060c0-40ae-4866-a4aa-6ee2ecf3728d')
print(os.listdir('.'))


['Lchi_zeros.pkl', '.prompts', '.config', 'engine-spec.md', '.kernel_llm_logs_1.txt', 'r14_lambda_min_table.csv', 'Lchi_zeros_neg.pkl', 'memory']


In [2]:
import pandas as pd
df = pd.read_csv('r14_lambda_min_table.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head(20))


(70, 7)
['function', 'T0', 'J', 'lambda_min', 'trace_residual', 'tr_Mz', 'tr_Ma']
 function T0 J lambda_min trace_residual tr_Mz tr_Ma
0 zeta 30.00 4 -5.045159e-15 -2.886580e-15 1.009446 1.009446
1 L_chi 30.00 4 -1.488630e-04 -5.343070e-04 4.434862 4.435397
2 zeta 30.00 8 -2.682885e-14 -1.421085e-14 2.238963 2.238963
3 L_chi 30.00 8 -1.633016e-04 -1.075858e-03 8.369126 8.370202
4 zeta 30.00 12 -1.075022e-13 -1.043610e-13 2.928659 2.928659
5 L_chi 30.00 12 -1.755441e-04 -1.624818e-03 12.130403 12.132028
6 zeta 30.00 16 -3.312915e-13 -4.307665e-13 4.212459 4.212459
7 L_chi 30.00 16 -1.868265e-04 -2.181359e-03 15.717532 15.719713
8 zeta 30.00 20 -5.507209e-13 -3.428369e-13 4.976926 4.976926
9 L_chi 30.00 20 -1.976095e-04 -2.745655e-03 20.360029 20.362774
10 zeta 30.00 24 -8.332093e-13 -3.428369e-13 5.669894 5.669894
11 L_chi 30.00 24 -2.081260e-04 -3.317890e-03 23.862475 23.865792
12 zeta 30.00 28 -3.310243e-12 5.423217e-12 6.376745 6.376745
13 L_chi 30.00 28 -2.185145e-04 -3.898250e-03 2

In [3]:
import pickle
with open('Lchi_zeros.pkl','rb') as f:
 zp = pickle.load(f)
with open('Lchi_zeros_neg.pkl','rb') as f:
 zn = pickle.load(f)
print(type(zp), len(zp), type(zp[0]))
print(zp[:3])
print(zn[:3])


<class 'list'> 130 <class 'str'>
['6.183578195450853914377517309708692525921500579228', '8.4572291744232307216053528627475187735407668860439', '12.674946417011355780482299145083092144682077684518']
['-4.1329037052128515950019193315655495660008212734511', '-9.4429311297285091171002621243109985512118800923338', '-11.282896441581600133225480792490362379842911812402']


In [4]:
df_orig = df[df.function=='L_chi'].copy()
print(df_orig.shape)
print(df_orig.head())
print("Unique T0:", df_orig.T0.unique())
print("Unique J:", df_orig.J.unique())


(35, 7)
 function T0 J lambda_min trace_residual tr_Mz tr_Ma
1 L_chi 30.0 4 -0.000149 -0.000534 4.434862 4.435397
3 L_chi 30.0 8 -0.000163 -0.001076 8.369126 8.370202
5 L_chi 30.0 12 -0.000176 -0.001625 12.130403 12.132028
7 L_chi 30.0 16 -0.000187 -0.002181 15.717532 15.719713
9 L_chi 30.0 20 -0.000198 -0.002746 20.360029 20.362774
Unique T0: [ 30. 46.13 60. 85.7 120. ]
Unique J: [ 4 8 12 16 20 24 28]


In [5]:
from e14c.todo import add_todos, TodoItem, update_todo
todos = [
 TodoItem(todo_id="1", content="Implement Hermite-Gauss basis and Fourier-side helpers", status="in_progress", priority="high"),
 TodoItem(todo_id="2", content="Implement M_zeros for L(chi) using cached zero lists (pos+neg)", status="pending", priority="high"),
 TodoItem(todo_id="3", content="Implement M_arith with corrected formula: -2 Re(chi(n)) Lambda(n)", status="pending", priority="high"),
 TodoItem(todo_id="4", content="Validate trace gate at one reference point", status="pending", priority="high"),
 TodoItem(todo_id="5", content="Run full grid (T0 x J) at sigma=1, X=1e9", status="pending", priority="high"),
 TodoItem(todo_id="6", content="Produce comparison table and final plot", status="pending", priority="high"),
]
state = add_todos(todos, None)


In [6]:
import numpy as np
import mpmath as mp
from scipy.special import gammaln, eval_genlaguerre, factorial
import math

# Hermite-Gauss basis:
# phi_n(t) = (1/sqrt(2^n n! sqrt(pi))) H_n(t) exp(-t^2/2)
# We use a real test function centered at T0 with width sigma:
# h_n(t) = (1/sqrt(sigma)) phi_n((t-T0)/sigma)
# Actually for the explicit formula we typically need an EVEN function in t-T0.
# But for the localized Weil quadratic form with arbitrary Hermite basis indexed by n=0..J-1,
# we may consider odd/even - the Hermite functions of both parities. The shifted test
# function is f_n(t) = phi_n((t-T0)/sigma) / sqrt(sigma) (L^2 normalized in t).
# For the Weil explicit formula we typically symmetrize: F_n(t) = h_n(t) + h_n(-t)
# Actually let me think more carefully.

# In a Weil-type explicit formula:
# sum_rho f(gamma) = arithmetic side
# where f is a real even Schwartz function with Fourier transform F supported as needed.
# But we want a quadratic form M[m,n] = sum_rho f_m_n(gamma) where f_m_n is a "product"-like
# kernel. Standard construction: take a test function family h_n and form the kernel
# k(gamma1, gamma2) = sum_n h_n(gamma1) h_n(gamma2). Then M_zeros[m,n] = sum_{rho1,rho2}
# but that gives PSD by construction.

# Actually the localized Weil quadratic form is defined by: pick g_n a basis of test functions,
# then for f = sum_n c_n g_n, the explicit formula gives a quadratic form in c. So:
# Q[m,n] = (zero sum side for g_m * g_n minus arith side for g_m*g_n? OR
# Q[m,n] = explicit-formula(g_m, g_n) where it's a bilinear form.

# The cleanest interpretation: define a real even Schwartz test function F (the convolution
# F = g * g*~). The Weil EF: sum_rho hat(F)(gamma) = arith side. The quadratic form is then
# obtained by taking F = (sum c_n g_n)(t) * (sum c_m g_m)(-t) integrated -- actually it's
# the "Beurling-Nyman" / Weil positivity setup.

# Concrete formulation used in Weil positivity tests (e.g., Bombieri):
# Let h(t) be a smooth function. Define the test pair (h, h_hat). The Weil EF is:
# sum_rho h_hat(gamma_rho) = h(0) log(pi^{-1}) - 2 sum_{n>=1} Lambda(n)/sqrt(n) * h(log n)
# + archimedean integral terms + (polar terms for zeta).
# For Weil positivity, one takes a function g and forms h = g * g^*(-x). Then h_hat = |g_hat|^2.
# RH equiv to: zero sum >= 0 for all g. The quadratic form Q[m,n] arises by taking g = sum c_n g_n,
# so h_hat = sum_{m,n} c_m c_n* hat(g_m) hat(g_n)* (assumed real => self-adjoint).

# For non-self-dual L-functions, the zero side and arith side need careful symmetrization.
# The specification says: prime sum should use -2 Re(chi(n)) Lambda(n)/sqrt(n) * h(log n)
# instead of -2 chi(n) Lambda(n)/sqrt(n) * h(log n) (or instead of -(chi(n)+chi-bar(n)) Lambda?)

# Per task: implement -2 * Re(chi(n)) * Lambda(n) / sqrt(n) * h(log n) as the prime-sum factor.
# We're told this is the analytical correction.

print("Setup complete")


Setup complete


In [7]:
# Define the engine. The spec says:
# - Test functions: Hermite-Gauss, centered at T0, width sigma, basis dim J.
# - We need M_zeros and M_arith such that Q = M_zeros - M_arith.
#
# Standard construction (consistent with the r14 magnitudes ~ tr ~ J and lambda_min ~ 1e-13 for zeta):
#
# Let psi_n(u) = phi_n(u) be normalized Hermite functions on R (in variable u).
# Test functions in t (the imaginary-part variable): 
# g_n(t) = (1/sqrt(sigma)) psi_n((t - T0)/sigma) for n=0..J-1.
# These are L^2(R)-orthonormal in t.
# 
# For the explicit formula we use functions on the spectral (gamma) side. Define
# h(gamma) = g_m(gamma) g_n(gamma) (a pointwise product), and require it to be
# the Fourier transform of some function on the "x" side (the log-prime variable).
# Actually that wouldn't be a Schwartz function generally.
#
# Alternative (and the standard one for "localized Weil quadratic form"):
# Take f(x) = sum_n c_n a_n(x), with a_n the Hermite functions in x scaled by sigma' 
# and modulated by exp(i T0 x). Then F(t) = hat(f)(t) is localized near T0.
# The Weil-positivity quadratic form is:
# Q[m,n] = (zero-side bilinear) - (arith-side bilinear).
# Specifically:
# M_zeros[m,n] = sum_rho conj(F_m(gamma_rho)) F_n(gamma_rho)
# = sum_rho conj(hat(a_m)(gamma_rho)) hat(a_n)(gamma_rho)
# where the basis a_n(x) are Hermite functions on the x-side modulated/translated.
#
# But trace of M_zeros must equal trace of M_arith. The trace gate is the diagonal
# explicit formula at each basis function (i.e., for the test function |F_n|^2 = |hat(a_n)|^2):
# sum_rho |hat(a_n)(gamma_rho)|^2 = arithmetic side(a_n, a_n)
# 
# Reference values:
# - For zeta at T0=46.13, J=12, sigma=1: tr(M_zeros) = 3.690677, looks small for J=12 zeros around T0=46.13.
# - Actually it's a sum over many (rho1,rho2) pairs? No, just diagonal explicit formula.

# Let me try a concrete construction:
# x-side basis: a_n(x) = phi_n(x/sigma) e^{i T0 x} / sqrt(sigma) (modulated Hermites)
# Then hat(a_n)(t) = sqrt(sigma) e^{... something ...} phi_n_hat(sigma (t - T0))
# Hermite functions are eigenfunctions of Fourier: hat(phi_n)(t) = (-i)^n phi_n(t).
# So hat(a_n)(t) = sqrt(sigma) (-i)^n phi_n(sigma(t - T0)) (up to convention).
# So |hat(a_n)(t)|^2 = sigma * phi_n(sigma(t-T0))^2.
# tr(M_zeros) = sum_n sum_rho sigma * phi_n(sigma(gamma-T0))^2
# For zeta at T0=46.13, sigma=1, J=12:
# tr_Mz = sum_{n=0..11} sum_rho phi_n(gamma-46.13)^2.
# Around gamma near 46.13, density ~ log(T)/(2pi). Each phi_n contributes ~1 (since
# phi_n^2 integrated is 1 and zero density times integral). Let's check numerically.

import mpmath as mp
mp.mp.dps = 30
# zeros of zeta near 46.13:
zeros = [float(mp.zetazero(n).imag) for n in range(1,40)]
print(zeros[:30])


[14.134725141734695, 21.022039638771556, 25.01085758014569, 30.424876125859512, 32.93506158773919, 37.586178158825675, 40.9187190121475, 43.327073280915, 48.00515088116716, 49.7738324776723, 52.970321477714464, 56.44624769706339, 59.34704400260235, 60.83177852460981, 65.1125440480816, 67.07981052949417, 69.54640171117398, 72.0671576744819, 75.70469069908393, 77.1448400688748, 79.33737502024937, 82.91038085408603, 84.73549298051705, 87.42527461312523, 88.80911120763446, 92.49189927055849, 94.65134404051989, 95.87063422824531, 98.83119421819369, 101.31785100573138]


In [8]:
import numpy as np
from numpy.polynomial.hermite import hermval

def phi_n_arr(n_max, x):
 """Return array shape (n_max+1, len(x)) of L^2-normalized Hermite functions phi_n(x).
 phi_n(x) = H_n(x) exp(-x^2/2) / sqrt(2^n n! sqrt(pi)).
 Use stable recurrence."""
 x = np.atleast_1d(x).astype(float)
 out = np.zeros((n_max+1, len(x)))
 out[0] = np.pi**(-0.25) * np.exp(-x*x/2)
 if n_max >= 1:
 out[1] = np.sqrt(2.0) * x * out[0]
 for n in range(1, n_max):
 out[n+1] = np.sqrt(2.0/(n+1)) * x * out[n] - np.sqrt(n/(n+1)) * out[n-1]
 return out

# Test orthonormality
xs = np.linspace(-30, 30, 20001)
P = phi_n_arr(10, xs)
G = P @ P.T * (xs[1]-xs[0])
print(np.diag(G)[:5], G[0,2], G[1,3])


[1. 1. 1. 1. 1.] -9.016020513443476e-17 -1.2719337026386075e-16


In [9]:
# Now let's check the trace of M_zeros for zeta to see if our framework matches r14_trace.
# tr(M_zeros) = sum_{n=0..J-1} sum_{rho} |hat(a_n)(gamma_rho)|^2
# With hat(a_n)(t) = sqrt(sigma) (-i)^n phi_n(sigma(t-T0)), the magnitude squared is
# sigma * phi_n(sigma(t-T0))^2.
# But we should sum over ALL zeros (positive and negative). For zeta, zeros come in conjugate pairs
# at +/- gamma. So actually rho = 1/2 + i*gamma with gamma in symmetric pairs.

# For zeta at T0=46.13, sigma=1, J=12: r14 says tr_Mz = 3.690677
# Let's compute (sigma=1):
T0 = 46.13
sigma = 1.0
J = 12

# Use a moderate number of positive zeros
N_zeros = 200
gammas_pos = np.array([float(mp.zetazero(n).imag) for n in range(1, N_zeros+1)])
# include negatives
gammas_all = np.concatenate([gammas_pos, -gammas_pos])

x_eval = sigma * (gammas_all - T0)
P = phi_n_arr(J-1, x_eval) # shape (J, 2*N_zeros)
weights = sigma * P**2 # |hat(a_n)|^2 at each gamma
tr_Mz = np.sum(weights)
print(f"tr_Mz = {tr_Mz}, reference = 3.690677")


tr_Mz = 3.6906769053295667, reference = 3.690677


In [10]:
# Excellent! Framework matches exactly. Now we need M_arith with the corrected formula.
# 
# Weil EF for L(s, chi) (chi primitive Dirichlet character mod q, conductor q):
# sum_rho h_hat(gamma_rho) 
# = (archimedean integral) - sum_{n>=2} [chi(n) Lambda(n)/sqrt(n)] h(log n)
# - sum_{n>=2} [chibar(n) Lambda(n)/sqrt(n)] h(log n)
# WAIT — actually for L(s, chi) the explicit formula is:
# sum_rho h_hat(gamma) = (arch part for chi) - sum_n chi(n) Lambda(n)/sqrt(n) h(log n)
# but the zero side is over zeros of L(s, chi) only (not L(s, chibar)).
#
# For Q (positivity quadratic form), we need a REAL self-adjoint form. The natural symmetrization is:
# F_n(x) = a_n(x), then h(x) = (F_m * F_n^reverse)(x) -- but this gets complicated.
#
# The cleanest "Weil positivity for L(s,chi)" sums over both L(s,chi) and L(s,chibar) zeros:
# Z(h) := sum_{rho of L(chi)} h_hat(gamma) + sum_{rho of L(chibar)} h_hat(gamma)
# = 2 * (arch part) - sum_n (chi(n)+chibar(n)) Lambda(n)/sqrt(n) h(log n)
# = 2 * (arch part) - 2 sum_n Re(chi(n)) Lambda(n)/sqrt(n) h(log n).
# This is the symmetrized version: arith side uses -2 Re(chi(n)) Lambda(n).
#
# Currently with original (presumed: just chi(n)) we use chi(n)+conj(chi(n)) on one side... 
# Actually let's just do the corrected version as specified.
#
# For our quadratic form:
# - M_zeros uses zeros of L(s,chi). The cached file gives zeros up to t=200 (and t=-200 for non-self-dual).
# - h(x) = a_m(x) * conj(a_n(-x)) (so h_hat(t) = conj(hat(a_m)) * hat(a_n) ? )
# Actually for a quadratic form M[m,n] we want:
# M_zeros[m,n] = sum_rho conj(hat(a_m)(gamma)) hat(a_n)(gamma)
# This is the sesquilinear form induced by point evaluation at zeros.
# With a_n(x) = (1/sqrt(sigma)) phi_n(x/sigma) e^{i T0 x}, hat(a_n)(t) = sqrt(sigma)(-i)^n phi_n(sigma(t-T0))
# (Convention: hat(f)(t) = integral f(x) e^{-i t x} dx... let me check sign carefully later.)
# Then M_zeros[m,n] = sigma * sum_rho (i^m (-i)^n) phi_m(sigma(g-T0)) phi_n(sigma(g-T0))
# = sigma * i^{m-n} sum_rho phi_m phi_n.
# For non-self-dual L(chi), the zeros gamma+ != -gamma-. Summing both pos and neg:
# M_zeros[m,n] = sigma * i^{m-n} [sum_{rho+} phi_m(sigma(g-T0)) phi_n(sigma(g-T0))
# + sum_{rho-} phi_m(sigma(g-T0)) phi_n(sigma(g-T0))]
#
# For the trace diagonal, m=n => i^{m-n}=1, so tr_Mz is real and positive.
# This matches the r14 framework.

# Now M_arith. For the Weil EF applied to test function h(x) corresponding to product 
# conj(a_m) * a_n (i.e. h_hat = conj(hat(a_m)) hat(a_n)) -- but this is in spectral side.
# 
# Equivalently: the test function on the x-side (log primes) for the bilinear form is
# k_{m,n}(x) = (a_m * a_n_tilde)(x), where a_n_tilde(x) = conj(a_n(-x)) ... actually
# convolution-flip: if h_hat(t) = conj(hat(a_m)(t)) hat(a_n)(t), then 
# h(x) = (conj(a_m)(-y) convolution a_n)(x) — too complex.
#
# Let me adopt a simpler approach. The trace gate says diagonal entries m=n equal arith side
# for test function |hat(a_m)|^2 on spectral side, which corresponds to test function
# h_m(x) = (a_m * a_m^reverse_conj)(x). 
#
# For off-diagonal M_arith[m,n], we use h_{m,n}(x) = (a_n * a_m^reverse_conj)(x).
# Since the trace gate is the critical thing, let's focus on getting trace right and using
# numpy double precision for matrix.

# A pragmatic implementation: compute hat(a_n)(t) directly, then "express" M_arith via
# the explicit formula written in spectral side:
# 
# Weil EF (for a real even test function h with hat h):
# sum_rho h_hat(gamma) = h_hat(... gamma=0 polar terms...) 
# + (1/(2pi)) integral h_hat(t) [Psi(gamma factor)] dt 
# - 2 sum_n Lambda(n)/sqrt(n) [Re(chi(n))] h(log n) <-- the corrected prime term
# where h(log n) = (1/(2pi)) integral h_hat(t) e^{i t log n} dt = (1/(2pi)) integral h_hat(t) n^{i t} dt.

# For the bilinear (sesquilinear) form, replace h_hat(t) with conj(hat(a_m))(t) * hat(a_n)(t).
# Then h(log n) = integral_t conj(hat(a_m))(t) hat(a_n)(t) e^{i t log n} dt / (2pi)
# = <hat(a_m), hat(a_n) e^{i t log n}> / (2pi)
# By Parseval: = <a_m, a_n shifted by log n>.

# This is getting complex. Let me follow the spec's explicit hint:
# "key implementation detail for the arithmetic matrix M_arith is the use of an analytic 
# Fourier transform for the Hermite-Gauss basis sum in the prime-power term, which involves 
# identities with Generalized Laguerre polynomials."
print("Continuing with analytic Fourier approach")


Continuing with analytic Fourier approach


In [11]:
# Hermite function identity:
# integral phi_m(u) phi_n(u) e^{i alpha u} du = ???
# Using the generating function or known identity:
# integral_R phi_m(u) phi_n(u) e^{i alpha u} du = 
# sqrt(pi/2) (something) ... let me derive.
#
# Actually a classical identity:
# integral phi_m(x) phi_n(x) e^{2 i s x} dx = e^{-s^2} (2 i s)^{|m-n|} sqrt(m_min! / m_max!) * L_{m_min}^{|m-n|}(2 s^2) * (some sign)
#
# Let me look up the Mehler-type identity.
#
# Actually one elegant identity (Klauder coherent-state matrix elements):
# <m | e^{i alpha (a + a^dag)/sqrt(2)} | n> = e^{-alpha^2/4} (i alpha/sqrt(2))^{n-m} sqrt(m!/n!) L_m^{n-m}(alpha^2/2) (for n>=m)
# Position operator x = (a + a^dag)/sqrt(2), so the matrix element of e^{i alpha x} in Hermite basis:
# <m | e^{i alpha x} | n> = integral phi_m(x) phi_n(x) e^{i alpha x} dx
# = e^{-alpha^2/4} (i alpha/sqrt(2))^{n-m} sqrt(m!/n!) L_m^{n-m}(alpha^2/2) for n>=m.

# So the integral S(m,n;alpha) := integral phi_m(x) phi_n(x) e^{i alpha x} dx is given by that formula.
# Let me verify numerically.

def S_exact(m, n, alpha):
 """Closed-form integral phi_m phi_n e^{i alpha x} dx."""
 if m > n:
 # Use symmetry: S(m,n,alpha) = conj(S(n,m,-alpha))? Actually integrand is symmetric in m,n. 
 # And the formula above is for n>=m. For m>n we swap.
 m, n = n, m
 diff = n - m # >=0
 # S = exp(-alpha^2/4) * (i alpha / sqrt(2))^diff * sqrt(m!/n!) * L_m^diff(alpha^2/2)
 from scipy.special import gammaln, eval_genlaguerre
 log_ratio = 0.5 * (gammaln(m+1) - gammaln(n+1))
 pref = np.exp(-alpha**2/4 + log_ratio) * (1j*alpha/np.sqrt(2))**diff
 Lag = eval_genlaguerre(m, diff, alpha**2/2)
 return pref * Lag

# Verify numerically:
xs = np.linspace(-30, 30, 100001)
dx = xs[1]-xs[0]
P = phi_n_arr(6, xs)
alpha = 0.7
for m,n in [(0,0),(0,1),(1,3),(2,5),(3,3)]:
 num = np.sum(P[m]*P[n]*np.exp(1j*alpha*xs)) * dx
 ex = S_exact(m, n, alpha)
 print(f"m={m},n={n}: num={num:.6g}, exact={ex:.6g}, diff={abs(num-ex):.2e}")


m=0,n=0: num=0.884706+0j, exact=0.884706+0j, diff=2.06e-12
m=0,n=1: num=0+0.437907j, exact=0+0.437907j, diff=1.02e-12
m=1,n=3: num=-0.243787+0j, exact=-0.243787+0j, diff=5.68e-13
m=2,n=5: num=0-0.121956j, exact=0-0.121956j, diff=2.84e-13
m=3,n=3: num=0.311935-6.82121e-17j, exact=0.311935+0j, diff=7.27e-13


In [12]:
# Great. Now let's set up the explicit formula carefully.
#
# CONVENTION for Fourier: Let hat(f)(t) = integral f(x) e^{-i t x} dx (physics convention).
# Then f(x) = (1/2pi) integral hat(f)(t) e^{i t x} dt.
#
# Hermite functions are Fourier eigenfunctions: hat(phi_n)(t) = sqrt(2*pi) (-i)^n phi_n(t).
# Wait — depends on normalization. Let me verify.
# Standard: integral phi_n(x) e^{-i t x} dx = sqrt(2 pi) (-i)^n phi_n(t).
xs = np.linspace(-25, 25, 100001)
dx = xs[1]-xs[0]
P = phi_n_arr(4, xs)
ts = np.array([0.0, 0.5, 1.0, 1.5])
for n in range(4):
 for t in ts:
 ft = np.sum(P[n] * np.exp(-1j*t*xs)) * dx
 exact = np.sqrt(2*np.pi) * (-1j)**n * phi_n_arr(n, np.array([t]))[n,0]
 # match
 print(f"n={n}: |ft - exact| at t=1: {abs(np.sum(P[n]*np.exp(-1j*1.0*xs))*dx - np.sqrt(2*np.pi)*(-1j)**n*phi_n_arr(n,np.array([1.0]))[n,0]):.2e}")


n=0: |ft - exact| at t=1: 2.66e-12
n=1: |ft - exact| at t=1: 3.76e-12
n=2: |ft - exact| at t=1: 1.88e-12
n=3: |ft - exact| at t=1: 1.54e-12


In [13]:
# Good. Now construct the engine carefully.
#
# Test function on x-side (logarithmic variable): 
# a_n(x) = (1/sqrt(sigma)) phi_n(x/sigma) e^{i T0 x}
# Then hat(a_n)(t) = integral a_n(x) e^{-i t x} dx = (1/sqrt(sigma)) integral phi_n(x/sigma) e^{-i (t - T0) x} dx
# = (1/sqrt(sigma)) * sigma * integral phi_n(u) e^{-i (t-T0) sigma u} du
# = sqrt(sigma) * sqrt(2pi) * (-i)^n * phi_n(sigma*(t-T0))
# So |hat(a_n)(gamma)|^2 = sigma * 2pi * phi_n(sigma(g-T0))^2.
# But we got tr_Mz = sum sigma * phi_n^2 = 3.69 earlier, not 2pi * sigma * sum phi_n^2.
# So there's a factor of 1/(2pi) somewhere convention. Let me redefine:
# M_zeros[m,n] = sigma * sum_rho i^{m-n} phi_m(sigma(g-T0)) phi_n(sigma(g-T0)) (without 2pi)
# This is what r14 used. So they're using a different normalization, namely the test function
# is "a_n / sqrt(2pi)" effectively. Fine.

# Let me just be consistent: drop the 2pi prefactor by using F_n(t) := sqrt(sigma) (-i)^n phi_n(sigma(t-T0))
# as the "spectral-side" basis, ignoring the Fourier sqrt(2pi) factor. Then:
# M_zeros[m,n] := sum_rho conj(F_m(gamma)) F_n(gamma)
# = sigma * i^{m-n} sum_rho phi_m(sigma(g-T0)) phi_n(sigma(g-T0))
# 
# Now M_arith. We need it via the Weil EF. The corresponding x-side test function (the one whose
# Fourier transform is conj(F_m) F_n) is:
# K_{m,n}(x) = (1/(2pi)) integral conj(F_m(t)) F_n(t) e^{i t x} dt.
# 
# Substituting F_n(t) = sqrt(sigma)(-i)^n phi_n(sigma(t-T0)):
# conj(F_m(t)) F_n(t) = sigma * i^m (-i)^n phi_m(sigma(t-T0)) phi_n(sigma(t-T0))
# = sigma * i^{m-n} phi_m * phi_n
# (assuming m,n real, phi real)
# 
# Then K_{m,n}(x) = (sigma/(2pi)) * i^{m-n} integral phi_m(sigma(t-T0)) phi_n(sigma(t-T0)) e^{i t x} dt
# = (sigma/(2pi)) * i^{m-n} * e^{i T0 x} * integral phi_m(sigma u) phi_n(sigma u) e^{i x u} du 
# (let t=T0+u)
# = (sigma/(2pi)) * i^{m-n} * e^{i T0 x} * (1/sigma) integral phi_m(v) phi_n(v) e^{i (x/sigma) v} dv
# = (1/(2pi)) * i^{m-n} * e^{i T0 x} * S(m, n; x/sigma)
# where S(m,n;alpha) is the integral we computed: 
# S(m,n;alpha) = e^{-alpha^2/4} (i alpha/sqrt(2))^{|n-m|} sqrt(min!/max!) L_min^{|n-m|}(alpha^2/2)
# (for n>=m, sign convention).
#
# So K_{m,n}(x) = (1/(2pi)) i^{m-n} e^{i T0 x} S(m,n; x/sigma).
#
# Weil EF for L(s, chi) (corrected, symmetric form):
# sum_rho conj(F_m(gamma)) F_n(gamma) 
# = (arch integral) + (polar?) - 2 sum_{n>=2} Re(chi(n)) Lambda(n)/sqrt(n) * K_{m,n}(log n)
# Wait — sign needs more thought. Let me be careful.
#
# Standard Weil EF (for completed L-function on critical line):
# sum_rho phi(gamma) = phi(...) + (arch) - 2 sum_n Lambda(n) cos(gamma log n)/sqrt(n) ... 
# Hmm depends on choice. Let me defer details and proceed.
#
# CRITICAL CHECK: for ζ at T0=46.13, J=12, sigma=1, r14 has tr(M_zeros) = 3.690677 and 
# trace residual ~ 0 (1e-13). So once we get the arith side right, traces match for ζ.

# Let me first build M_zeros for ζ and L_chi using the simple construction, then build M_arith
# using the analytic Fourier and verify trace gate against ζ.

def build_Mzeros(gammas_pos, gammas_neg, T0, sigma, J):
 """gammas_pos, gammas_neg: arrays of positive (and negative) imaginary parts of zeros.
 For zeta, gammas_neg should be -gammas_pos. For L_chi (non-self-dual) they differ.
 M[m,n] = sigma * i^{m-n} sum_rho phi_m(sigma(g-T0)) phi_n(sigma(g-T0)).
 """
 all_g = np.concatenate([gammas_pos, gammas_neg])
 u = sigma * (all_g - T0)
 P = phi_n_arr(J-1, u) # (J, Nzeros)
 # M = sigma * i^{m-n} * (P @ P.T)
 PP = (P @ P.T) * sigma
 powers = (1j) ** np.arange(J)
 M = np.outer(powers, np.conj(powers)) * PP # i^m * (-i)^n * PP = i^{m-n}*PP
 return M

# Check trace for ζ:
import mpmath as mp
mp.mp.dps = 30
T0=46.13; sigma=1.0; J=12
N_zeros = 200
gammas_pos = np.array([float(mp.zetazero(n).imag) for n in range(1, N_zeros+1)])
gammas_neg = -gammas_pos.copy()
Mz = build_Mzeros(gammas_pos, gammas_neg, T0, sigma, J)
print("tr(M_z) =", Mz.trace().real, " (ref 3.690677)")
print("imag part of trace:", Mz.trace().imag)


tr(M_z) = 3.690676905329566 (ref 3.690677)
imag part of trace: 0.0


In [14]:
# Trace matches. Now build M_arith. We use:
# M_arith[m,n] = (Polar terms) + (Arch integral) - 2 sum_{n_p>=2} Re(chi(n_p)) Lambda(n_p)/sqrt(n_p) * K_{m,n}(log n_p)
# (for ζ, replace Re(chi(n_p)) with 1).
#
# Wait — for ζ the prime-side coefficient is just Lambda(n)/sqrt(n) times some factor of 2 due to 
# pairing with complex-conjugate zeros. The standard Weil EF for ζ:
# sum_rho phi(gamma) = phi(i/2)+phi(-i/2) + (arch term) - 2 sum_n Lambda(n)/sqrt(n) phi_hat(log n) ... wait
# I need to think which convention is being used here.
#
# Let me take a pragmatic empirical approach: pick the form
# M_arith[m,n] = (Arch+Polar) - C * sum_p prime_factor * K_{m,n}(log p^k)
# and tune C and the formula to make trace match for ζ at the reference point.
# 
# Actually the r14 table tells us trace residual is ~1e-13 (numerical floor) for ζ. So the formula 
# they used works. Let me reconstruct:
# tr(M_arith) for ζ at T0=46.13, J=12, sigma=1 = 3.690677.
# tr(M_zeros) = 3.690677 (above).
# We need tr(M_arith) = 3.690677 within 1e-13.
#
# Standard Weil EF (Bombieri form) for L(s) on critical line (real form):
# sum_gamma h(gamma) = h_hat(0) log(condcond) - 2 sum_{n>=2} Lambda(n) h_hat(log n) / sqrt(n)
# + (Polar) + 2 Re integral_{-inf..inf} h(t) [-Gamma'/Gamma + log pi]... 
# Hmm different conventions. Let me just use what's known to give trace match.

# A clean version (Weil 1952): for completed Lambda(s) = (q/pi)^(s/2) Gamma(s/2) zeta(s),
# sum_{rho} h_hat(gamma_rho) = h(0) + h_hat(0)?... no.
# OK let me just try concrete construction step by step.
#
# Actually here is the Weil EF for ζ (from Iwaniec-Kowalski, ch. 5):
# Let F be even Schwartz on R, and phi(s) = integral F(x) e^{(s-1/2)x} dx for s on critical line: phi(1/2+it) = hat F(-t).
# Then:
# sum_{rho} phi(rho) = phi(0) + phi(1) - sum_{n>=1} Lambda(n)/sqrt(n) (F(log n) + F(-log n))
# + (1/(2 pi)) integral F-hat(t) (Psi(1/4 + it/2) - log pi) dt [archimedean]
# where Psi = Gamma'/Gamma.
# 
# rho = 1/2 + i gamma, so phi(rho) = hat(F)(-gamma) but F is even so hat(F)(-gamma) = hat(F)(gamma).
# In our notation hat(F)(t) is real-valued for F even real.
#
# For our quadratic form, we have phi(rho) = conj(F_m(gamma)) F_n(gamma) (not from an even real F).
# Let's allow complex F and write h(t) := conj(F_m(t)) F_n(t). The x-side "test function" is then 
# K_{m,n}(x) = (1/2pi) integral h(t) e^{i t x} dt as before.
#
# For ζ the EF in this notation (assuming our sign conventions): I need to figure out which sign.
# Let me try the form:
# sum_rho conj(F_m) F_n at gamma 
# = K_{m,n}(0) * 2 (the polar terms at s=0,1 contribute K_{m,n}(0)? not quite)
# + (arch integral)
# - sum_{n>=2} Lambda(n)/sqrt(n) [K_{m,n}(log n) + K_{m,n}(-log n)]
# 
# Actually for non-self-dual L(chi): no polar terms (since L(chi) is entire for non-trivial chi).
# Plus the arith side becomes:
# - sum_{n>=2} Lambda(n)/sqrt(n) [chi(n) K_{m,n}(log n) + chibar(n) K_{m,n}(-log n)]
# OK so the original (uncorrected) formula likely used [chi(n) K(log n) + chibar(n) K(-log n)] 
# which is NOT generally real-valued in the bilinear form, leading to the trace error.
# The CORRECTED version (per task) uses -2 Re(chi(n)) Lambda(n) * (K(log n) + K(-log n))/2 
# = -Re(chi(n)) Lambda(n) [K(log n) + K(-log n)]/sqrt(n)? Or simply -2 Re(chi(n)) Lambda(n) K(log n) / sqrt(n)?
# 
# Per the task: "the prime sum should be proportional to Re(chi(n))Lambda(n)" and 
# "implement a version of the arithmetic side where the prime sum uses -2 * Re(chi(n)) instead of 
# the formulation that produced the previously observed error".
# 
# I'll interpret as: replace [chi(n) K(log n) + chibar(n) K(-log n)] with 
# 2 Re(chi(n)) [K(log n) + K(-log n)] / 2 = Re(chi(n)) [K(log n) + K(-log n)]? 
# OR equivalently: -2 sum_n Re(chi(n)) Lambda(n)/sqrt(n) * Re(K(log n))? 
# 
# Let me think: for self-dual case (chi real), chi = chibar, so the original formula already gives 
# [chi(n)(K(log n) + K(-log n))]. For non-self-dual, we'd want a symmetric version.
# 
# I'll use this form for M_arith (which generalizes ζ's symmetric form):
# M_arith[m,n] = (Arch[m,n]) - sum_{n_p>=2} (chi_term)_p Lambda(p^k)/sqrt(p^k) * [K_{m,n}(log p^k) + K_{m,n}(-log p^k)]
# where for ζ: chi_term = 1; for L_chi (corrected): chi_term = Re(chi(n_p)).
# This makes "prime sum term coefficient = -2*Re(chi(n))*Lambda(n)/sqrt(n)" if we symmetrize the 
# K's into 2*Re(K), or just splits as above.
# 
# Let me just check this gives trace match for ζ first.

print("Setting up arith side")


Setting up arith side


In [15]:
# K_{m,n}(x) = (1/(2pi)) i^{m-n} e^{i T0 x} S(m,n; x/sigma)
# K_{m,n}(0) = (1/(2pi)) i^{m-n} S(m,n; 0)
# S(m,n; 0) = delta_{m,n} (orthonormality, since e^{0}=1 and phi_m phi_n integrated = delta).
# So K_{m,n}(0) = delta_{m,n}/(2pi).

# K_{m,n}(log p^k) + K_{m,n}(-log p^k) = (1/(2pi)) i^{m-n} [e^{i T0 log p^k} S(m,n; log p^k / sigma) + e^{-i T0 log p^k} S(m,n; -log p^k / sigma)]
# Note S(m,n; -alpha) = (let's check from formula).
# S(m,n;alpha) with m<=n, diff=n-m: = e^{-alpha^2/4} (i alpha/sqrt(2))^diff sqrt(m!/n!) L_m^diff(alpha^2/2).
# S(m,n;-alpha) = e^{-alpha^2/4} (-i alpha/sqrt(2))^diff sqrt(m!/n!) L_m^diff(alpha^2/2) = (-1)^diff S(m,n;alpha) = (-1)^{n-m} S(m,n;alpha).

# Now archimedean term. For ζ:
# Arch_zeta(F) = (1/(2pi)) integral F-hat(t) (Re Psi(1/4 + it/2) - 1/2 log pi) dt
# but we have h(t) = conj(F_m) F_n. Let's use the dual representation.
#
# In our formulation we want the bilinear form:
# Arch[m,n] = (1/(2pi)) integral_R h(t) g_arch(t) dt
# where g_arch(t) is the spectral-side gamma-factor density:
# for ζ: g_arch(t) = Re[Psi(1/4 + i t/2)] - (1/2) log pi
# for L(chi) chi odd: g_arch(t) = Re[Psi(3/4 + i t/2)] - (1/2) log pi (since gamma factor is Gamma((s+1)/2))
# for L(chi mod 5): chi(−1) = chi(4) = -1, so chi is ODD => gamma factor Gamma_R(s+1) = pi^{-(s+1)/2} Gamma((s+1)/2).
# 
# Actually more carefully: the local archimedean factor is Gamma_R(s+a) with a=0 (even) or a=1 (odd).
# For chi mod 5 with chi(-1)=chi(4)=-1, chi is odd, so a=1.
# 
# The full completed L-function: Lambda(s,chi) = (q/pi)^{(s+a)/2} Gamma((s+a)/2) L(s,chi),
# with q the conductor. For chi mod 5 primitive, q=5, a=1.
# 
# Functional equation: Lambda(s, chi) = epsilon * Lambda(1-s, chibar).
# 
# Weil EF for L(s, chi) (non-self-dual): sum over zeros of L(s,chi) only:
# sum_rho phi(rho) = - sum_n Lambda(n) (chi(n) F(log n) + chibar(n) F(-log n))/sqrt(n)
# + Arch integral involving Gamma'/Gamma((s+a)/2)
# + log(q/pi)/2 * F-hat(0)
# 
# When symmetrizing (summing over zeros of L(chi) AND L(chibar)), we get -2 Re(chi(n)) on arith side.
# But the spec says only the zeros of L(chi). So if we only use L(chi) zeros on the LEFT side,
# and use -2 Re(chi(n)) on the RIGHT, we are NOT consistent.
# 
# Hmm — unless we are working in a setting where: since chi has order 4 and chi != chibar, 
# our M_zeros uses zeros of L(s,chi) only. And the EF for L(s,chi) alone uses chi(n) (not Re(chi(n))).
# Symmetrization (using Re(chi)) requires summing over both L(chi) and L(chibar) zeros.
#
# Since we have only Lchi_zeros.pkl (and conjugate via Lchi_zeros_neg), we don't have L(chibar) zeros 
# explicitly. BUT: zeros of L(s,chibar) = conjugate of zeros of L(s,chi) under reflection. Specifically,
# if rho = 1/2 + i*gamma is a zero of L(s,chi), then conj(rho) = 1/2 - i*gamma is a zero of L(s,chibar).
# So zeros of L(chibar) are at gamma -> -gamma_chi (with gamma_chi the zeros of L(chi)).
# 
# So: the set {zeros of L(chi)} U {zeros of L(chibar)} = {gamma_chi} U {-gamma_chi} = all positive and 
# negative imaginary parts of L(chi) zeros = exactly what we already have in Lchi_zeros + Lchi_zeros_neg 
# (as zeros of L(chi)) PLUS the negatives = -Lchi_zeros (zeros of L(chibar) at +T) and -Lchi_zeros_neg.
#
# Hmm, so if "M_zeros" uses zeros of L(chi) (positive + negative imag), then to match the 
# "Re(chi)" symmetric arith side, M_zeros should sum over zeros of L(chi) AND L(chibar).
# Zeros of L(chibar) = -(zeros of L(chi)). So we'd add to M_zeros the contributions at -gamma 
# for each gamma in L(chi). The L(chi) zeros are at {6.18..., 8.46..., 12.67..., ...} pos and 
# {-4.13..., -9.44..., -11.28...} neg. L(chibar) zeros are at the negatives of these: 
# {-6.18..., -8.46..., ...} pos-of-Lchibar (these are imaginary parts of L(chibar) zeros).
# 
# So combined zero set is: ±6.18, ±8.46, ..., ±4.13, ±9.44, ...
# = {Lchi_pos_imag} U {-Lchi_pos_imag} U {Lchi_neg_imag} U {-Lchi_neg_imag}
# = symmetric set.
# 
# So we should DOUBLE the zero side using ±gammas symmetrically. Let me modify the M_zeros 
# computation for L_chi to include both signs of all 260 zeros.

print("Understanding: for symmetric (corrected) formula, M_zeros sums over zeros of L(chi) AND L(chibar)")
print("which equals using both g and -g for every g in Lchi_zeros pos+neg union.")


Understanding: for symmetric (corrected) formula, M_zeros sums over zeros of L(chi) AND L(chibar)
which equals using both g and -g for every g in Lchi_zeros pos+neg union.


In [16]:
# BUT — the original (r14) formula also presumably uses zeros of L(chi) only. The trace residual 
# is "systematic ~1e-4". If we now sum over zeros of both L(chi) and L(chibar), tr(M_zeros) DOUBLES,
# but then arith side also needs to double (use -2 Re(chi) instead of -chi).
#
# Looking at r14: at T0=46.13, J=12, L_chi: tr_Mz = 13.543189, tr_Ma = 13.543868, residual = -6.79e-4.
# Compare to ζ at same point: tr_Mz = 3.690677 — so L_chi has tr_Mz roughly 3-4x larger.
# That's because L_chi has more zeros in the relevant window (and we're using pos+neg).
# 
# Let me see: with pos+neg L_chi zeros (260 total), what is tr_Mz?

import pickle
with open('Lchi_zeros.pkl','rb') as f:
 Lchi_pos = [float(s) for s in pickle.load(f)]
with open('Lchi_zeros_neg.pkl','rb') as f:
 Lchi_neg = [float(s) for s in pickle.load(f)]
gpos = np.array(Lchi_pos)
gneg = np.array(Lchi_neg)
print("Number positive zeros:", len(gpos), "range:", gpos.min(), gpos.max())
print("Number negative zeros:", len(gneg), "range:", gneg.min(), gneg.max())

# tr_Mz using only L(chi) zeros (pos+neg as in cache):
T0=46.13; sigma=1.0; J=12
all_g = np.concatenate([gpos, gneg])
u = sigma*(all_g - T0)
P = phi_n_arr(J-1, u)
tr_Mz_simple = (sigma * (P*P).sum())
print(f"tr_Mz (zeros of L(chi) only, pos+neg) = {tr_Mz_simple:.6f}")
print(f"r14 reference tr_Mz = 13.543189")

# Now symmetrized (zeros of L(chi) AND L(chibar)):
all_g_sym = np.concatenate([gpos, gneg, -gpos, -gneg])
u = sigma*(all_g_sym - T0)
P = phi_n_arr(J-1, u)
tr_Mz_sym = sigma*(P*P).sum()
print(f"tr_Mz (symmetric, both L(chi) and L(chibar)) = {tr_Mz_sym:.6f}")


Number positive zeros: 130 range: 6.1835781954508535 199.7728627790239
Number negative zeros: 130 range: -200.06783291852213 -4.132903705212851
tr_Mz (zeros of L(chi) only, pos+neg) = 6.557401
r14 reference tr_Mz = 13.543189
tr_Mz (symmetric, both L(chi) and L(chibar)) = 13.543189


In [17]:
# Perfect! r14 was already using the SYMMETRIC zero set. So tr_Mz = 13.543189 corresponds to 
# summing over zeros of L(chi) AND L(chibar) (i.e., using ±gamma for every gamma in pos cache).
#
# So M_zeros (in r14, also for our corrected version) = sum over symmetric zero set.
# The original (r14) arith side had a residual ~6.79e-4 (out of 13.54), suggesting it was using 
# chi(n) (not Re(chi(n))) which gives the wrong real coefficient.
#
# The CORRECTED version: M_arith uses -2 Re(chi(n)) Lambda(n)/sqrt(n) * [K(log n) + K(-log n)] / 2 ?
# Or directly: -2 Re(chi(n)) Lambda(n)/sqrt(n) * (K(log n) + K(-log n)) / 2 with appropriate 
# sign convention.
# 
# Since the symmetric zero side gives tr_Mz = 13.54 and matches the r14 reference, the correct
# arith side that matches it (when correctly using Re(chi)) should give the same value.
#
# Let me build M_arith using the EF:
# M_arith[m,n] = (log(q/pi)) * K_{m,n}(0) # from log(condcond)/2 * 2 (since q/pi^a contribution)
# + Arch_int[m,n] # gamma factor archimedean
# - sum_{p^k} 2 Re(chi(p^k)) Lambda(p^k)/sqrt(p^k) * [K_{m,n}(log p^k) + K_{m,n}(-log p^k)]/2
# 
# Hmm I'll write the symmetric EF and verify trace match for ζ first.
#
# Symmetric EF for ζ (sum over zeros at ±gamma, where each gamma > 0 is a zero):
# Standard form (Weil): for even Schwartz F,
# sum_{gamma} F(gamma) [sum over all zeros, including conjugates] 
# = 2 F-hat(0) Re(...) + arch integral - 2 sum_n Lambda(n)/sqrt(n) F-hat(log n)
# where F-hat(x) = (1/2pi) integral F(t) e^{-i t x} dt or with another convention.
# 
# Wait I keep flip-flopping conventions. Let me just write it directly and test.
# 
# Spectral side (zeros side) for ζ: sum_rho conj(F_m(gamma)) F_n(gamma) where gammas are all 
# imaginary parts (pos + neg) of zeta zeros. With F_n(t) = sqrt(sigma) (-i)^n phi_n(sigma(t-T0)):
# Already shown tr matches r14 = 3.690677.

# For arith side construction, use the bilinear form K_{m,n}(x). The diagonal of K at x=0:
# K_{m,m}(0) = (1/2pi) * S(m,m;0) = (1/2pi) * 1 = 1/(2pi).
# So diagonal of "polar/conductor" term: log(q/pi^a) * K_{m,m}(0) = log(q/pi^a) / (2pi). 
# For ζ this is log(1/pi) / (2pi) = -log(pi)/(2pi).
# Tr from polar (for J=12): J * log(1/pi)/(2pi)
zeta_polar_diag = -np.log(np.pi)/(2*np.pi) * 12
print("ζ polar trace contribution = ", zeta_polar_diag)
# For ζ there's also a pole at s=1 giving +F(0) where F = some related thing. Actually for ζ EF:
# sum_rho phi(rho) = phi(0) + phi(1) + ...
# In our K-formulation: phi(rho)=conj(F_m(gamma))F_n(gamma); at rho=0 (s=0): rho corresponds to gamma=-i/2,
# at rho=1: gamma=+i/2. These are off the real line. Evaluating F at i/2 is complex.
# 
# This is getting messy. Let me take a fundamentally different approach.
print("Switching to a Mellin/log-variable construction that's well-known.")


ζ polar trace contribution = -2.186273038055438
Switching to a Mellin/log-variable construction that's well-known.


In [18]:
# Let me re-anchor on what we know:
# r14 zeta tr_Mz = 3.69... AND tr_Ma matches to 1e-13.
# So whatever formula r14 uses for M_arith DOES give trace match for zeta.
# 
# For L_chi r14 used some non-symmetric/wrong factor giving residual 1e-4.
# The corrected version should swap to symmetric factor and trace should match to 1e-13.
# 
# Rather than reverse-engineer all of r14's conventions, let me use a clean, well-defined explicit 
# formula and ensure trace matches for ζ as a calibration. Once that's done, I'll apply the 
# corrected formula for L_chi.

# I'll use Mestre-style sum (see Mestre 1986 / Conrey "L-functions and random matrix theory"):
# For an L-function L(s, f) with completed Lambda(s,f) = (q/pi^d)^{s/2} prod Gamma((s+kappa_j)/2) * L(s,f),
# zeros are 1/2 + i*gamma. The explicit formula:
# sum_{gamma} h(gamma) = (1/2pi) integral_{-inf}^{inf} h(t) [log(q/pi^d) + sum_j (Psi((1/4 + kappa_j/2 + it/2)) + Psi((1/4 + kappa_j/2 - it/2)))/2] dt 
# - 2 sum_{n>=1} Lambda_f(n) / sqrt(n) * h-check(log n)
# where h-check(x) = (1/2pi) integral h(t) cos(t x) dt? or e^{i t x}? Depends.
# 
# Let me be VERY concrete and use the Tao/Iwaniec-Kowalski Theorem 5.12 explicit formula. 
# Or just write it once and verify by computation.

# Form 1 (Iwaniec-Kowalski Th 5.12):
# Let F: R -> C be smooth, of compact support OR rapid decay, and let
# F-hat(s) := integral F(x) e^{(s-1/2) x} dx (Mellin-like)
# Then for L(s,f) (entire, satisfying functional equation), with Lambda(s,f) = gamma_inf(s) L(s,f), gamma_inf(s) = (q/pi^d)^{s/2} prod Gamma((s+kappa_j)/2):
# sum_rho F-hat(rho) = sum_n a_f(n) F(log n)/sqrt(n) + (epsilon) sum_n conj(a_f(n)) F(-log n)/sqrt(n) + Arch(F)
# where epsilon is root number, a_f(n) = Lambda_f(n) (von Mangoldt for L), etc.
# 
# For our purpose: it's easier to use a parameterization in terms of test function h on real line 
# (gammas) directly.

# Form 2 (cleanest for our purposes): Use convention that
# sum_{rho of L(s,f)} h(gamma_rho) = h_hat(0) * log(condcond) + arch + arith
# where 
# arith = - sum_{n>=2} [a_f(n) h_hat(log n) + conj(a_f(n)) h_hat(-log n)] / sqrt(n)
# h_hat(x) = (1/2pi) integral h(t) e^{-i t x} dt (cosine transform of even part)
# a_f(n) = Lambda(n) chi(n) for L(s, chi)
# a_f(n) = Lambda(n) for ζ
# 
# Yes this is the form. For self-dual: a_f(n) = conj(a_f(n)), so arith = -2 sum a_f(n) Re(h_hat(log n))/sqrt(n). 
# For non-self-dual (chi != chibar), arith retains both terms.
# 
# Now, when we have a sesquilinear form M[m,n] = sum_rho conj(F_m) F_n, set h(t) = conj(F_m)(t) F_n(t).
# Note h is COMPLEX in general. The EF still applies (linearly).
# Then h_hat(x) = (1/2pi) integral h(t) e^{-i t x} dt = K_{m,n}(-x).
# Using K_{m,n}(x) defined above as (1/2pi) integral h(t) e^{i t x} dt, so h_hat(x) = K_{m,n}(-x).
# 
# So arith for L(s,chi) =
# M_arith[m,n] = K_{m,n}(0) * log(q/pi^d) + Arch_int[m,n] 
# - sum_{n>=2} Lambda(n)/sqrt(n) * [chi(n) K_{m,n}(-log n) + chibar(n) K_{m,n}(log n)]
# 
# CORRECTED version (symmetric, sum over zeros of L(chi) AND L(chibar)) — since we are doubling 
# the zero side, we double the arith side and use Re(chi):
# M_arith[m,n] = 2 * [K_{m,n}(0) * log(q/pi^d) + Arch_int[m,n]] 
# - 2 sum_{n>=2} Re(chi(n)) Lambda(n)/sqrt(n) * [K_{m,n}(-log n) + K_{m,n}(log n)]
# 
# Hmm but the doubling of arch + polar is the doubling for both L(chi) and L(chibar) contributions.
# Since chi and chibar have same conductor and same archimedean factor (since chi(-1) = -1 = chibar(-1)),
# the arch+polar piece for L(chibar) is identical, so it's just 2× the single-EF contribution.
# 
# So for L_chi (mod 5), q=5, d=1, kappa = 1 (chi odd).

print("Formulation set. Now implementing.")


Formulation set. Now implementing.


In [19]:
# First implement K_{m,n}(x) using analytic Fourier (Laguerre identity)
from scipy.special import gammaln, eval_genlaguerre

def K_matrix(x, T0, sigma, J):
 """Return JxJ complex matrix K[m,n](x) = (1/2pi) i^{m-n} e^{i T0 x} S(m,n; x/sigma).
 S(m,n;alpha) = e^{-alpha^2/4} (i alpha/sqrt(2))^{|m-n|} sqrt(min!/max!) L_{min}^{|m-n|}(alpha^2/2)
 """
 alpha = x / sigma
 # Compute S(m,n;alpha) matrix
 # For each (m,n): let mn=min, mx=max, diff=mx-mn
 mn_arr = np.minimum.outer(np.arange(J), np.arange(J))
 mx_arr = np.maximum.outer(np.arange(J), np.arange(J))
 diff = mx_arr - mn_arr
 log_ratio = 0.5*(gammaln(mn_arr+1) - gammaln(mx_arr+1))
 # eval_genlaguerre vectorized:
 # we need L_{mn}^{diff}(alpha^2/2) for each (m,n).
 Lag = np.zeros((J,J))
 arg = alpha**2/2
 for i in range(J):
 for j in range(J):
 Lag[i,j] = eval_genlaguerre(mn_arr[i,j], diff[i,j], arg)
 S = np.exp(-alpha**2/4 + log_ratio) * (1j*alpha/np.sqrt(2))**diff * Lag
 # K_{m,n}(x) = (1/2pi) i^{m-n} e^{i T0 x} S(m,n; x/sigma)
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 pref = np.outer(pow_m, pow_n_conj) # i^m * (-i)^n = i^{m-n}
 K = (1.0/(2*np.pi)) * pref * np.exp(1j*T0*x) * S
 return K

# Quick verification at x=0:
K0 = K_matrix(0.0, T0=46.13, sigma=1.0, J=6)
print("K(0):")
print(np.round(K0.real, 6))
print("Off-diagonal max:", np.max(np.abs(K0 - np.diag(np.diag(K0)))))
print("Diagonal value:", K0[0,0], "Expected: 1/(2pi)=", 1/(2*np.pi))


K(0):
[[ 0.159155 0. -0. 0. 0. 0. ]
 [ 0. 0.159155 0. -0. 0. 0. ]
 [-0. 0. 0.159155 0. -0. 0. ]
 [ 0. -0. 0. 0.159155 0. -0. ]
 [ 0. 0. -0. 0. 0.159155 0. ]
 [ 0. 0. 0. -0. 0. 0.159155]]
Off-diagonal max: 0.0
Diagonal value: (0.15915494309189535+0j) Expected: 1/(2pi)= 0.15915494309189535


In [20]:
# Good. Now build M_arith for ζ and verify trace match.
# For ζ: q=1, d=1, kappa=0 (since ζ is "even"), polar terms at s=0,1.
# Plus we should sum over zeros of ζ (with conjugates): ±gammas.
# 
# EF for ζ (with both ±gamma summation):
# sum_{gammas, +/-} h(gamma) = 2 [ Polar terms + Arch_int - sum a(n) Lambda(n)/sqrt(n) (K(-log n) + K(log n)) ]
# Wait — when we sum over both gamma>0 and gamma<0 zeros of ζ, the spectral side counts each pair once.
# A single EF (over all zeros of ζ counted once) already includes ±gammas.
# 
# So for ζ:
# M_zeros[m,n] = sum_{rho of zeta} conj(F_m(gamma)) F_n(gamma) [sum over all ±gamma]
# M_arith[m,n] = K_{m,n}(0) * log(1/pi) + Arch_int[m,n] - sum_{n_int>=2} Lambda(n)/sqrt(n) [K_{m,n}(-log n) + K_{m,n}(log n)]
# + (polar terms from poles at s=0,1)
# 
# Polar terms for ζ: the EF says sum_rho h(gamma) = h(i/2) + h(-i/2) + ... where the +/- refer to 
# poles at s=1 (which is rho=1) and s=0 (which is rho=0). In gamma = -i*(s-1/2), s=1 -> gamma = -i/2;
# s=0 -> gamma = i/2.
# So polar = h(i/2) + h(-i/2) = conj(F_m(i/2)) F_n(i/2) + conj(F_m(-i/2)) F_n(-i/2).
# But our F_n is defined on REAL gamma. We need to extend analytically:
# F_n(t) = sqrt(sigma) (-i)^n phi_n(sigma(t-T0)).
# phi_n is entire (Hermite polynomial * Gaussian).
# F_n(i/2) = sqrt(sigma) (-i)^n phi_n(sigma*(i/2 - T0)) -- complex valued.

# This is getting heavy. Let me consider a different angle. The spec says:
# "engine reference behavior at T0=85.7, sigma=2, J=10 ... ζ gives |λ_min|/tr ~ 1e-8 to 1e-10 (numerical floor)"
# So whatever they did, it included all needed terms for ζ.
#
# But for trace gate alone, we just need tr(M_arith) = tr(M_zeros). And tr operates over diagonal m=n.
# Let me focus on the DIAGONAL: K_{m,m}(0) = 1/(2pi), so polar/conductor diag = log(q/pi^d)/(2pi).
# Arch diag = (1/2pi) integral |F_m(t)|^2 * gamma_arch_density(t) dt.
# Prime diag = -2 sum_n Lambda(n) Re(a(n)) /sqrt(n) * Re(K_{m,m}(log n)) [taking real part for diagonal]
# where for self-dual a(n) = chi(n) = real, and for non-self-dual symmetric form a(n) = Re(chi(n))
# 
# K_{m,m}(x) = (1/2pi) e^{i T0 x} S(m,m; x/sigma) -- and S(m,m;alpha) is real (diff=0, so (i alpha)^0 =1):
# S(m,m;alpha) = e^{-alpha^2/4} L_m(alpha^2/2)
# So K_{m,m}(x) = (1/2pi) e^{i T0 x} e^{-(x/sigma)^2/4} L_m((x/sigma)^2/2)
# K_{m,m}(log n) + K_{m,m}(-log n) = (1/2pi) e^{-(log n)^2/(4 sigma^2)} L_m((log n)^2/(2 sigma^2)) * 2 cos(T0 log n)
# 
# So diagonal prime sum for ζ:
# = - sum_n Lambda(n)/sqrt(n) [K(log n) + K(-log n)]
# = - (1/pi) sum_n Lambda(n)/sqrt(n) e^{-(log n)^2/(4 sigma^2)} L_m((log n)^2/(2 sigma^2)) cos(T0 log n)
# 
# For corrected L(chi):
# = - (1/pi) sum_n Re(chi(n)) Lambda(n)/sqrt(n) e^{-(log n)^2/(4 sigma^2)} L_m((log n)^2/(2 sigma^2)) cos(T0 log n)
# (and 2x this if we double the EF since we sum over both L(chi) and L(chibar) zeros)
# 
# Wait, the spec says: "the prime sum uses -2 * Re(chi(n))". The factor "-2" is in the coefficient,
# which is what we already have (because of the symmetrized [K(log n) + K(-log n)] and the symmetric 
# zero sum). Let me just implement and check.

# Let me first calibrate on ζ. Compute tr(M_arith) for ζ at T0=46.13, sigma=1, J=12, X=10^5
# and see if it matches tr(M_zeros) = 3.690677.

from sympy import primerange

def von_mangoldt_terms(X):
 """Return list of (n, Lambda(n)) for n=p^k <= X."""
 out = []
 primes = list(primerange(2, X+1))
 for p in primes:
 pk = p
 logp = math.log(p)
 while pk <= X:
 out.append((pk, logp))
 pk *= p
 return out

import math
X = 10**5
vm = von_mangoldt_terms(X)
print("Number of (p^k <= X) terms:", len(vm))


Number of (p^k <= X) terms: 9700


In [21]:
# Let me compute the diagonal trace for ζ.
# Need arch integral for diagonal: 
# Arch_diag_m = (1/2pi) integral |F_m(t)|^2 g_arch(t) dt
# where |F_m(t)|^2 = sigma * phi_m(sigma(t-T0))^2
# For ζ: g_arch(t) = Re[Psi(1/4 + i t/2)] - (1/2) log pi
# But careful — log(q/pi) constant part might already be absorbed into "polar/conductor" term.
# 
# Let me use the most common form: (Iwaniec-Kowalski (5.27)):
# Sum_rho h(gamma) = (1/2pi) integral h(t) [log q - d log pi + sum_j Re Psi(1/4 + kappa_j/2 + it/2)] dt
# + (poles) - 2 Re sum_{n>=2} Lambda(n) chi(n) n^{-1/2} hat(h)(log n)
# Hmm with self-dual convention. For non-self-dual:
# - sum_n Lambda(n)/sqrt(n) [chi(n) hat(h)(log n) + chibar(n) hat(h)(-log n)]
# where hat(h)(x) = (1/2pi) integral h(t) e^{-i t x} dt ... no this is getting too convention-dependent.

# Let me numerically validate: compute tr(M_arith) for ζ using:
# tr_Marith = log(q/pi^d)/(2pi) * J 
# + sum_{m=0}^{J-1} Arch_int_diag_m 
# - sum_{n>=2} Lambda(n)/sqrt(n) * sum_m [K_{m,m}(log n) + K_{m,m}(-log n)]
# = J * log(q/pi^d)/(2pi) 
# + sum_m (1/2pi) int |F_m(t)|^2 g_arch(t) dt (where g_arch = sum_j Re Psi(1/4 + kappa_j/2 + it/2))
# - (1/pi) sum_{n>=2} Lambda(n)/sqrt(n) e^{-(log n)^2/(4 sigma^2)} cos(T0 log n) * sum_m L_m((log n)^2/(2 sigma^2))
# 
# For ζ: q=1, kappa_j = {0}, d=1, so log(q/pi^d) = -log pi.
# 
# Sum of Laguerres: sum_{m=0}^{J-1} L_m(y). 
# We have an identity: sum_{m=0}^{J-1} L_m(y) = L_{J-1}^{(1)}(y) (Christoffel-Darboux for Laguerre).
# Yes! sum_{m=0..J-1} L_m^{(0)}(y) = L_{J-1}^{(1)}(y).
# 
# Let me verify this:
from scipy.special import eval_laguerre, eval_genlaguerre
y = 1.7
J_test = 6
s1 = sum(eval_laguerre(m, y) for m in range(J_test))
s2 = eval_genlaguerre(J_test-1, 1, y)
print(s1, s2) # Should match


-0.912796416666666 -0.912796416666667


In [22]:
# Great. Now let me compute tr(M_arith) for ζ.
import numpy as np
import mpmath as mp
from sympy import primerange
import math

def prime_powers_up_to(X):
 """Generator of (n=p^k, log p) for prime powers <= X."""
 out_n = []
 out_logp = []
 out_p = []
 for p in primerange(2, X+1):
 logp = math.log(p)
 pk = p
 while pk <= X:
 out_n.append(pk)
 out_logp.append(logp)
 out_p.append(p)
 pk *= p
 return np.array(out_n), np.array(out_logp), np.array(out_p)

X = 10**5
ns_pp, logp_pp, p_pp = prime_powers_up_to(X)
log_n = np.log(ns_pp)
sqrt_n = np.sqrt(ns_pp)
Lambda = logp_pp

print(f"Num prime powers <= {X}: {len(ns_pp)}")

def tr_M_arith_zeta(T0, sigma, J, X_data=None):
 """tr(M_arith) for ζ."""
 if X_data is None:
 ns, logp, _ = prime_powers_up_to(10**5)
 else:
 ns, logp, _ = X_data
 log_n = np.log(ns)
 Lambda = logp
 # Conductor term: J * (-log pi) / (2pi)
 cond_term = J * (-np.log(np.pi)) / (2*np.pi)
 # Arch integral diag (for ζ): sum_m (1/2pi) integral |F_m(t)|^2 [Re Psi(1/4 + i t/2)] dt
 # |F_m(t)|^2 = sigma * phi_m(sigma(t-T0))^2
 # Change vars u = sigma(t-T0), t = T0 + u/sigma, dt = du/sigma:
 # = (1/2pi) integral phi_m(u)^2 Re Psi(1/4 + i (T0 + u/sigma)/2) du
 # Sum over m=0..J-1 of phi_m(u)^2 has nice closed form via Christoffel-Darboux for Hermites:
 # sum_{m=0}^{J-1} phi_m(u)^2 = phi_{J-1}(u)^2 * something... actually it's:
 # sum_{m=0}^{J-1} phi_m(u)^2 = sqrt(J/2) [phi_J(u) phi_{J-1}'(u) - phi_J'(u) phi_{J-1}(u)] / something... 
 # Simpler: just compute the sum directly with quadrature.
 # Quadrature: Gauss-Hermite or trapezoidal on [-R, R] in u.
 R = max(20.0, 6*np.sqrt(J))
 npts = 4001
 u = np.linspace(-R, R, npts)
 du = u[1]-u[0]
 P = phi_n_arr(J-1, u) # (J, npts)
 sumphi2 = (P*P).sum(axis=0) # (npts,)
 t_vals = T0 + u/sigma
 # Re Psi(1/4 + i t/2) for ζ archimedean
 rePsi = np.array([float(mp.re(mp.psi(0, mp.mpc(0.25, t/2)))) for t in t_vals])
 arch_diag = (1/(2*np.pi)) * np.sum(sumphi2 * rePsi) * du
 # Prime term diagonal: 
 # = - sum_n Lambda(n)/sqrt(n) * sum_m [K_{m,m}(log n) + K_{m,m}(-log n)]
 # K_{m,m}(x) = (1/2pi) e^{i T0 x} e^{-(x/sigma)^2/4} L_m((x/sigma)^2/2)
 # sum: (1/pi) e^{-(x/sigma)^2/4} cos(T0 x) * sum_m L_m((x/sigma)^2/2) = (1/pi) e^{-y/2} cos(T0 x) L_{J-1}^{(1)}(y)
 # where y = (x/sigma)^2/2.
 y = (logp/sigma)**2 / 2.0 # wait should use log_n = log(p^k)
 y = (log_n/sigma)**2 / 2.0
 fac = (1.0/np.pi) * np.exp(-y/2) * np.cos(T0 * log_n) * eval_genlaguerre(J-1, 1, y)
 prime_diag = - np.sum(Lambda / np.sqrt(ns) * fac)
 # Polar terms for ζ: poles at s=0,1: h(i/2)+h(-i/2). Since h = |F_m|^2 for m fixed, sum over m.
 # h_m(t) = |F_m(t)|^2 = sigma * phi_m(sigma(t-T0))^2 at t = ±i/2.
 # phi_m(z) is entire. Compute:
 polar = 0.0
 for sgn in [1, -1]:
 z = sigma * (sgn*0.5j - T0) # argument: sigma*(t-T0) at t = sgn*i/2
 # Compute phi_m(z) for m=0..J-1 using Hermite polynomial recurrence (complex):
 phivals = np.zeros(J, dtype=complex)
 phivals[0] = np.pi**(-0.25) * np.exp(-z*z/2)
 if J >= 2:
 phivals[1] = np.sqrt(2.0) * z * phivals[0]
 for n in range(1, J-1):
 phivals[n+1] = np.sqrt(2.0/(n+1)) * z * phivals[n] - np.sqrt(n/(n+1)) * phivals[n-1]
 # |F_m(t)|^2 -- but at complex t, |.|^2 isn't standard. Actually for analytic continuation,
 # h(z) extends to conj(F_m(z*)) F_m(z) for real z, but for complex z need to be careful.
 # In Weil EF, the test function h is defined on real axis; polar terms are h evaluated at 
 # specific complex points via analytic continuation. The analytic continuation of 
 # h(t) = |F_m(t)|^2 (defined for real t) is NOT |F_m(z)|^2 but rather the unique entire 
 # extension. For F_m real-on-real, h_m(z) = F_m(z)^2. But F_m(t) = sqrt(sigma)(-i)^m phi_m(sigma(t-T0)),
 # phi_m real for real arg => F_m(t) = sqrt(sigma)(-i)^m * (real). For real t, F_m is purely real * (-i)^m,
 # so |F_m(t)|^2 = sigma * phi_m^2 (real). The analytic continuation: ext to z by 
 # h_m(z) = sigma * phi_m(sigma(z-T0))^2 (treat as polynomial-times-Gaussian).
 polar += sigma * (phivals**2).sum()
 return cond_term + arch_diag + prime_diag + polar

import time
t0=time.time()
tr_Ma_zeta = tr_M_arith_zeta(46.13, 1.0, 12, (ns_pp, logp_pp, p_pp))
print(f"tr_Marith for ζ at T0=46.13, sigma=1, J=12: {tr_Ma_zeta}")
print(f"tr_Mzeros reference: 3.690677")
print(f"residual: {tr_Ma_zeta - 3.690677}")
print(f"time: {time.time()-t0:.2f}s")


Num prime powers <= 100000: 9700


tr_Marith for ζ at T0=46.13, sigma=1, J=12: (3.6904538618672893+0j)
tr_Mzeros reference: 3.690677
residual: (-0.0002231381327106874+0j)
time: 1.59s


In [23]:
# Close but not at numerical floor — 2e-4 residual. Likely an issue with one of:
# (a) the arch integral integration accuracy (sum_phi^2 has rapid oscillations from Psi for J=12)
# (b) the polar terms convention
# (c) missing factor / normalization
# 
# Let me also test at larger X to ensure the prime cutoff isn't the issue:
# The r14 already showed X=1e5 works for sigma>=2 but FAILS for sigma=1 J>20 with large neg lambda_min.
# Here we have J=12 < 20, so X=1e5 might be marginal.

X = 10**6
ns_pp6, logp_pp6, _ = prime_powers_up_to(X)
print(f"Num prime powers <= {X}: {len(ns_pp6)}")
tr_Ma_zeta_X6 = tr_M_arith_zeta(46.13, 1.0, 12, (ns_pp6, logp_pp6, None))
print(f"X=1e6: tr_Ma = {tr_Ma_zeta_X6}, residual = {tr_Ma_zeta_X6 - 3.690677}")


Num prime powers <= 1000000: 78734


X=1e6: tr_Ma = (3.690677116669486+0j), residual = (1.1666948607214067e-07+0j)


In [24]:
# X=1e6 gives 1e-7, X=1e5 gives 2e-4. Confirms we need X=1e9 for high precision at sigma=1.
# At X=1e9 the prime sum will have ~5e7 terms — let's see if we can compute that efficiently.
# 
# Also let me check the polar+arch sign for accuracy. With X=1e6 residual is 1e-7 — could be 
# coming from the arch integral quadrature too. Let me also use higher accuracy arch integral.

# Test with X=1e7 first for speed
X = 10**7
ns_pp7, logp_pp7, _ = prime_powers_up_to(X)
print(f"Num prime powers <= {X}: {len(ns_pp7)}")
tr_Ma_zeta_X7 = tr_M_arith_zeta(46.13, 1.0, 12, (ns_pp7, logp_pp7, None))
print(f"X=1e7: residual = {tr_Ma_zeta_X7 - 3.690677}")


Num prime powers <= 10000000: 665134


X=1e7: residual = (-9.466809336444726e-08+0j)


In [25]:
# 1e-7 at X=1e7. The residual probably alternates due to oscillations of cos(T0 log n). 
# The arch integral with 4001 trapezoidal points may also be limiting. Let me improve arch integral.

# But first — for trace gate at sigma=1 the spec says X=1e9 is needed. Let's not worry about 
# matching to 1e-13 perfectly yet — focus on getting things working consistently, then run X=1e9.
# 
# Actually let me think about runtime. X=1e9 gives ~5e7 prime powers. For each grid point (T0, J), 
# we need to compute prime sum involving log(n), sqrt(n), cos(T0 log n), Laguerre at (log n/sigma)^2/2.
# 
# Per (T0, J): about ~5e7 array operations with numpy. That's ~5 seconds with good vectorization.
# Grid: 5 T0 * 7 J = 35 points. So 35 * 5s = 3 min. Plus generating prime powers up to 1e9 takes 
# significant memory (~400MB for arrays) and time.
# 
# Crucial optimization: prime powers <= 1e9 — that's 50e6 prime numbers, ~50M ints. About 400MB 
# RAM. Let's verify feasibility and consider alternative (do prime sums in chunks).

# Let me estimate time to enumerate primes up to 1e9:
import time
# sympy primerange is slow. Use sieve.
# Use numpy sieve for primes up to N.
def sieve_primes(N):
 sieve = np.ones(N+1, dtype=bool)
 sieve[:2] = False
 for i in range(2, int(N**0.5)+1):
 if sieve[i]:
 sieve[i*i::i] = False
 return np.nonzero(sieve)[0]

t0=time.time()
primes_1e7 = sieve_primes(10**7)
print(f"primes <= 1e7: {len(primes_1e7)}, time {time.time()-t0:.1f}s")

# For 1e9, sieve uses about 1GB memory. Let's avoid full sieve. We'll process in chunks.


primes <= 1e7: 664579, time 0.0s


In [26]:
# Wow numpy is fast! 1e7 in 0s. Try 1e9:
t0=time.time()
primes_1e8 = sieve_primes(10**8)
print(f"primes <= 1e8: {len(primes_1e8)}, time {time.time()-t0:.1f}s")


primes <= 1e8: 5761455, time 0.7s


In [27]:
# OK 1e8 in 0.7s. For 1e9 it'll be ~7s and ~1GB. Let's try a segmented sieve approach to save memory.
# Actually, let me see if 1e9 fits.

# Memory: bool array of 1e9 is 1GB. That might be too much. Let's try uint8 packed bits via bitarray
# OR use a segmented sieve. Or compute the prime sum in segments.

# Simpler: use segmented sieve to enumerate primes and accumulate the prime sum chunk by chunk.

def segmented_prime_sum(N, sigma, T0, J, chi_n_func, segment_size=10**7):
 """Compute, for each m in 0..J-1:
 S_m = sum_{p^k <= N} chi_n_func(p^k) Lambda(p^k)/sqrt(p^k) * factor(p^k, m)
 where factor depends on (log p^k)/sigma etc.
 
 Actually we want a scalar (the trace) which is sum over m of factor. So let's compute that.
 For trace: factor_m(n) = (1/pi) e^{-(log n)^2/(4 sigma^2)} cos(T0 log n) L_m(y) where y=(log n/sigma)^2/2.
 Sum over m: (1/pi) e^{-y/2} cos(T0 log n) L_{J-1}^{(1)}(y).
 
 Returns: scalar trace prime contribution = - sum chi_n(p^k) Lambda(p^k)/sqrt(p^k) * (1/pi) e^{-y/2} cos(T0 log p^k) L_{J-1}^{(1)}(y)
 """
 # Use segmented sieve
 sieve_init = np.ones(int(np.sqrt(N))+2, dtype=bool)
 sieve_init[:2] = False
 for i in range(2, int(np.sqrt(len(sieve_init)))+1):
 if sieve_init[i]: sieve_init[i*i::i] = False
 base_primes = np.nonzero(sieve_init)[0]
 
 total = 0.0
 low = 2
 while low <= N:
 high = min(low + segment_size - 1, N)
 size = high - low + 1
 seg = np.ones(size, dtype=bool)
 for p in base_primes:
 if p*p > high: break
 start = max(p*p, ((low + p - 1)//p)*p)
 seg[start - low::p] = False
 primes_seg = np.nonzero(seg)[0] + low
 # contribute p^1 term
 logp = np.log(primes_seg).astype(np.float64)
 log_n = logp
 n = primes_seg.astype(np.float64)
 sqn = np.sqrt(n)
 chi_vals = chi_n_func(primes_seg)
 if np.any(chi_vals != 0):
 y = (log_n/sigma)**2 / 2.0
 fac = (1.0/np.pi) * np.exp(-y/2) * np.cos(T0 * log_n) * eval_genlaguerre(J-1, 1, y)
 contrib = - np.sum(chi_vals * logp / sqn * fac)
 total += contrib
 # contribute prime powers p^k for k>=2 (only need to consider primes_seg with p^k <= N)
 # For primes in the segment, also include p^2, p^3, ... if p^k <= N
 for p_seg in primes_seg:
 pk = p_seg * p_seg
 while pk <= N:
 logp_val = np.log(p_seg)
 log_n_val = np.log(pk)
 sqn_val = np.sqrt(pk)
 chi_val = chi_n_func(np.array([pk]))[0]
 if chi_val != 0:
 y = (log_n_val/sigma)**2 / 2.0
 fac = (1.0/np.pi) * np.exp(-y/2) * np.cos(T0 * log_n_val) * eval_genlaguerre(J-1, 1, y)
 total += - chi_val * logp_val / sqn_val * fac
 pk *= p_seg
 low = high + 1
 return total

# Quick test for ζ at X=1e7
import time
t0=time.time()
def chi_zeta(n_arr): return np.ones_like(n_arr, dtype=np.float64)
prime_sum_test = segmented_prime_sum(10**7, 1.0, 46.13, 12, chi_zeta, segment_size=10**6)
print(f"prime sum (X=1e7, segmented): {prime_sum_test}, time {time.time()-t0:.1f}s")
# Compare with direct
ns7, lpp7, _ = ns_pp7, logp_pp7, None
log_n = np.log(ns7); fac_dir = (1.0/np.pi) * np.exp(-(log_n/1.0)**2/4) * np.cos(46.13*log_n) * eval_genlaguerre(12-1, 1, (log_n/1.0)**2/2)
prime_sum_dir = -np.sum(lpp7/np.sqrt(ns7)*fac_dir)
print(f"prime sum (direct): {prime_sum_dir}")


prime sum (X=1e7, segmented): -0.11405484179701082, time 0.2s
prime sum (direct): -0.11405484179701084


In [28]:
# Excellent! Segmented sieve matches direct. But for the FULL M_arith (not just trace), we need
# off-diagonal entries too. The formula uses sum over m of L_m factor for the trace — but for 
# off-diagonal we need the full K_{m,n}(log p^k). That's more expensive.
#
# Strategy: for each prime power, compute K_{m,n}(log p^k) JxJ matrix and accumulate.
# Per prime power: JxJ = 12x12 = 144 ops. For X=1e9: 5e7 * 144 = 7e9 ops — too slow.
# 
# Better: vectorize over many primes at once. Compute K matrix dependence on log n in batch.
# 
# For each prime power n, K_{m,n}(log n) involves: e^{i T0 log n} (scalar), L_min^diff at arg=(log n)^2/(2 sigma^2),
# and (i alpha/sqrt(2))^diff where alpha=log n/sigma. Plus e^{-alpha^2/4}.
# 
# Vectorize: process batches of N_batch primes at a time, for each (m,n) compute the JxJ matrix 
# elements as length-N_batch arrays. The Laguerre evaluations are the bottleneck.
# 
# For J=28 (max), JxJ=784 entries. Each Laguerre eval at degree up to ~27. With scipy 
# eval_genlaguerre vectorized over n, this is doable.
# 
# Let me prototype: process all primes <= 1e9 in batches of 1e6, accumulate JxJ K_{m,n}(log n)*Lambda(n)/sqrt(n)*chi(n)
# Plus also K_{m,n}(-log n) part.
# 
# Note K_{m,n}(-x) = (1/2pi) i^{m-n} e^{-i T0 x} S(m,n; -x/sigma).
# S(m,n; -alpha) = (-1)^{|m-n|} S(m,n; alpha) (from formula since (i*(-alpha))^diff = (-1)^diff (i alpha)^diff).
# So K_{m,n}(-x) = (1/2pi) i^{m-n} e^{-i T0 x} (-1)^{|m-n|} S(m,n; x/sigma)
# 
# Therefore [K_{m,n}(log n) + K_{m,n}(-log n)] = (1/2pi) i^{m-n} S(m,n; log n / sigma) * [e^{i T0 log n} + (-1)^{|m-n|} e^{-i T0 log n}]
# 
# For diff even: bracket = 2 cos(T0 log n)
# For diff odd: bracket = 2i sin(T0 log n)
# 
# Useful for understanding structure, not necessarily for speed. Let's just compute directly.

# Now let me think about: do I even need the FULL M_arith with X=1e9, or can I evaluate the off-diagonal
# at X=1e5/1e6 and only do trace gate at X=1e9?
# 
# The spec says: "For wide localizations (σ ≥ 2), a prime cutoff of X=10⁵ is sufficient ... 
# However, for narrower localizations (σ ≤ 1), this cutoff is insufficient for J>20, causing 
# large, spurious negative λ_min values. For ζ, this artifact is confirmed to be resolved by 
# using a larger cutoff of X=10⁹."
# So we need X=1e9 for the FULL M_arith at sigma=1.
# 
# Time budget: 3600s. Let me estimate full M_arith at X=1e9.
# Number of primes <= 1e9: ~50.8M. Number of prime powers (with k>=2): about sqrt(1e9)=31623 primes 
# contribute squares, plus a few hundred cubes etc. So total ~50.8M + ~32k ≈ 50.8M.
# Per prime: compute JxJ K-matrix. JxJ = 28x28 = 784 entries.
# Vectorized batch of 1e6 primes: compute S(m,n;alpha) for 28*28=784 (m,n) pairs and 1e6 alphas.
# That's 7.8e8 numbers, each involving eval_genlaguerre. May be slow with scipy.
# 
# Alternative: store base data and only use 1e9 cutoff for trace gate. For off-diagonals use smaller
# cutoff (since the matrix algebra is in double precision anyway).
# 
# Actually the analytical correction is what we're testing. If trace gate passes for L_chi with 
# corrected formula, that's most of the evidence. The lambda_min just confirms convergence.
# 
# Plan:
# - For TRACE: use X=1e9, segmented prime sum (compute only trace contribution — fast).
# - For OFF-DIAG: use X=1e5 (matches r14 default). The matrix algebra will have spurious effects
# at sigma=1, J>20, BUT that's the SAME artifact for original and corrected formulas. The 
# COMPARISON is fair if we use same X for off-diag in both.

# Actually wait — r14 used X=1e9 for everything (per task description). Let me reconsider.
# Task: "the same grid used in the decisive r16 experiment ... σ=1, prime cutoff X=10⁹".
# So we should use X=1e9 for the full M_arith.
# 
# Let me estimate carefully. For J=28, computing K matrix for one prime takes ~784 ops with Laguerres.
# Vectorized over batch of 1e6 primes:
# - For each (m,n) with diff=|m-n|, compute L_min^{diff}(alpha^2/2) over 1e6 alphas.
# - Use np.polynomial.laguerre or scipy.special.eval_genlaguerre vectorized over alpha.

# Let me time it:
import time
N_test = 10**6
alpha_test = np.random.uniform(0, 10, N_test)
arg_test = alpha_test**2 / 2
t0=time.time()
for i in range(28):
 for j in range(28):
 mn = min(i,j); diff = abs(i-j)
 Lag = eval_genlaguerre(mn, diff, arg_test)
print(f"784 calls of eval_genlaguerre on 1e6 points: {time.time()-t0:.2f}s")


784 calls of eval_genlaguerre on 1e6 points: 27.00s


In [29]:
# 27s per 1e6 primes for J=28. For 5e7 primes: 27 * 50 = 1350s = 22.5 min PER (T0, J=28) point.
# Times 5 T0 values = 112 min just for J=28. Too slow.
# 
# Optimization: use Laguerre recurrence instead of repeated calls.
# L_n^(alpha)(x) recurrence:
# (n+1) L_{n+1}^a(x) = (2n+1+a-x) L_n^a(x) - (n+a) L_{n-1}^a(x)
# 
# We can compute all (m,n) pairs in one pass over batch.

# Actually for our case: for each (m,n) pair we have m_min = min(m,n), diff = |m-n|.
# So we need L_{m_min}^{diff}(arg) for all (m,n) with m,n in 0..J-1.
# Equivalent: for each (mn, diff) with mn+diff <= J-1, compute L_{mn}^{diff}(arg).
# Total distinct (mn, diff) pairs: J*(J+1)/2 ≈ J^2/2. For J=28: 406 pairs.
# 
# Better: pre-compute, for each diff in 0..J-1, the array of L_0^diff, L_1^diff, ..., L_{J-1-diff}^diff
# all at the same arg vector using the recurrence. This is much faster.

def all_laguerre_batch(args, J):
 """Compute Lag[diff, mn, :] = L_{mn}^{(diff)}(args) for mn+diff <= J-1, args is 1D array.
 Returns 3D array of shape (J, J, N) where Lag[diff, mn, :] is the value (other entries are 0).
 """
 N = len(args)
 Lag = np.zeros((J, J, N))
 for diff in range(J):
 # Recurrence for L_n^diff:
 # L_0^a = 1
 # L_1^a = 1 + a - x
 # L_{n+1}^a = ((2n+1+a-x) L_n - (n+a) L_{n-1}) / (n+1)
 a = diff
 n_max = J - 1 - diff
 Lag[diff, 0] = 1.0
 if n_max >= 1:
 Lag[diff, 1] = 1.0 + a - args
 for n in range(1, n_max):
 Lag[diff, n+1] = ((2*n+1+a-args) * Lag[diff, n] - (n+a) * Lag[diff, n-1]) / (n+1)
 return Lag

import time
t0=time.time()
J_test=28
arg_test = np.random.uniform(0, 10, 10**6)
Lag = all_laguerre_batch(arg_test, J_test)
print(f"all_laguerre_batch J={J_test}, N=1e6: {time.time()-t0:.2f}s")

# Verify against scipy:
from scipy.special import eval_genlaguerre
arg_small = np.array([0.5, 1.2, 3.7])
Lag_small = all_laguerre_batch(arg_small, 5)
for d in range(5):
 for mn in range(5-d):
 ref = eval_genlaguerre(mn, d, arg_small)
 print(f"diff={d},mn={mn}: ours={Lag_small[d,mn]}, ref={ref}, diff={np.max(np.abs(Lag_small[d,mn]-ref))}")


all_laguerre_batch J=28, N=1e6: 4.77s
diff=0,mn=0: ours=[1. 1. 1.], ref=[1. 1. 1.], diff=0.0
diff=0,mn=1: ours=[ 0.5 -0.2 -2.7], ref=[ 0.5 -0.2 -2.7], diff=0.0
diff=0,mn=2: ours=[ 0.125 -0.68 0.445], ref=[ 0.125 -0.68 0.445], diff=4.440892098500626e-16
diff=0,mn=3: ours=[-0.14583333 -0.728 1.99283333], ref=[-0.14583333 -0.728 1.99283333], diff=1.1102230246251565e-16
diff=0,mn=4: ours=[-0.33072917 -0.5456 1.3103375 ], ref=[-0.33072917 -0.5456 1.3103375 ], diff=4.440892098500626e-16
diff=1,mn=0: ours=[1. 1. 1.], ref=[1. 1. 1.], diff=0.0
diff=1,mn=1: ours=[ 1.5 0.8 -1.7], ref=[ 1.5 0.8 -1.7], diff=0.0
diff=1,mn=2: ours=[ 1.625 0.12 -1.255], ref=[ 1.625 0.12 -1.255], diff=2.220446049250313e-16
diff=1,mn=3: ours=[ 1.47916667 -0.608 0.73783333], ref=[ 1.47916667 -0.608 0.73783333], diff=4.440892098500626e-16
diff=2,mn=0: ours=[1. 1. 1.], ref=[1. 1. 1.], diff=0.0
diff=2,mn=1: ours=[ 2.5 1.8 -0.7], ref=[ 2.5 1.8 -0.7], diff=0.0
diff=2,mn=2: ours=[ 4.125 1.92 -1.955], ref=[ 4.125 1.92 -1.955], 

In [30]:
# Great — 5s per 1e6 for J=28. So 5e7 primes / 1e6 = 50 batches * 5s = 250s per (T0, J=28) point.
# 5 T0 * (4..28 in steps of 4) = 5*7 = 35 points. Mean J=16, time scales as J^2/2, so avg time
# (16/28)^2 = 0.33 * 250 = ~80s, times 35 points = ~2800s. Cutting it close but doable.
# 
# Actually most expensive: we don't need to recompute the prime sum for each (T0, J). We can:
# (a) Run primes once, accumulate K_{m,n} contributions for J=28 (largest); 
# (b) For smaller J, just use submatrix.
# (c) But different T0 need different cos/sin and e^{i T0 log n}. So per T0, we have to redo.
# 
# Optimization: For each batch of primes, compute all needed (m,n) for J=28 and ALL 5 T0 values 
# simultaneously. The Laguerre and exp(-alpha^2/4) parts are T0-independent. Only the e^{i T0 log n}
# depends on T0. So we can amortize.

# Let me build the engine carefully now. Define a function that computes the FULL M_arith matrix
# for given (T0, sigma, J, X, chi_func) using segmented sieve.

def compute_M_arith(T0, sigma, J, X, chi_n_func, q, kappa, polar_func=None, 
 arch_npts=4001, arch_R=None, segment_size=2*10**6, verbose=False):
 """
 Compute M_arith[m,n] = K(0)*log(q/pi^d) (diag-only via K(0) delta) + Arch_int[m,n] 
 - sum_{n_int>=2} Lambda(n)/sqrt(n) * 
 [chi(n) K_{m,n}(-log n) + chibar(n) K_{m,n}(log n)] 
 (or for corrected symmetric: -2*Re(chi(n)) [K(log n)+K(-log n)]/2 ... see)
 For now, use the FORM determined by chi_n_func:
 - chi_n_func returns either chi(n) (original) or 2*Re(chi(n)) (corrected SYMMETRIC factor)
 The prime sum is: - sum_n chi_n_func(n) Lambda(n)/sqrt(n) * [K_{m,n}(log n) + K_{m,n}(-log n)]/2
 
 Actually we have two options:
 (A) Original (non-self-dual): - sum [chi(n) K(-log n) + chibar(n) K(log n)] (different K args!)
 (B) Corrected (symmetric): - sum 2*Re(chi(n)) [K(log n) + K(-log n)]/2 = - sum Re(chi(n)) [K(log n)+K(-log n)]
 
 To be flexible: pass chi_func that returns coefficient c(n), and use the formula:
 - sum_n c(n) Lambda(n)/sqrt(n) * 0.5 * [K(log n) + K(-log n)] for "symmetric"
 - sum_n [c1(n) K(-log n) + c2(n) K(log n)] Lambda(n)/sqrt(n) for "original"
 """
 pass # implementation below

# Let me just write two separate functions: one for ζ (real factor), one for L_chi (complex/real).
# 
# For ζ (already working with diagonal trace test, scalar chi_n=1):
# prime contribution to M[m,n] = - sum_n Lambda(n)/sqrt(n) [K_{m,n}(log n) + K_{m,n}(-log n)]
# For L_chi CORRECTED (symmetric over zeros of L(chi) and L(chibar)):
# contribution = - sum_n 2*Re(chi(n)) Lambda(n)/sqrt(n) * [K_{m,n}(log n) + K_{m,n}(-log n)]/2
# = - sum_n Re(chi(n)) Lambda(n)/sqrt(n) * [K_{m,n}(log n) + K_{m,n}(-log n)]
# Plus DOUBLE the arch+conductor terms (since both L(chi) and L(chibar) contribute identically).
# 
# For L_chi ORIGINAL (non-symmetric, just L(chi) zeros):
# prime contribution = - sum_n Lambda(n)/sqrt(n) [chi(n) K_{m,n}(-log n) + chibar(n) K_{m,n}(log n)]
# Plus arch+cond term (just from L(chi)).

# But in r14, tr(M_zeros) = 13.54 corresponded to the symmetric zero sum. So r14 must have done
# either symmetric arith OR doubled. Yet r14 has 1e-4 residual. So r14 likely used a slightly 
# different formula (maybe asymmetric arith but symmetric zeros, or vice versa).

# Regardless, our job is to implement the CORRECTED version (symmetric on both sides) and show
# it gives trace floor.

print("Implementation plan set.")


Implementation plan set.


In [31]:
# Let me implement the full M_arith computation. The strategy:
# - Build M_arith as JxJ complex matrix.
# - Iterate over prime powers in segments. For each batch:
# 1. Compute alpha = log(n)/sigma for each n in batch.
# 2. Compute Lag[diff, mn, batch] using recurrence.
# 3. For each (m,n) pair: S(m,n;alpha) = exp(-alpha^2/4) (i alpha/sqrt(2))^{|m-n|} sqrt(min!/max!) L_{min}^{|m-n|}(alpha^2/2)
# 4. K_pos = (1/2pi) i^{m-n} e^{i T0 log n} S; K_neg = (1/2pi) i^{m-n} e^{-i T0 log n} (-1)^{|m-n|} S
# 5. Sum = [K_pos + K_neg]
# 6. Accumulate -sum_n weight(n) Lambda(n)/sqrt(n) Sum into M_arith[m,n], where weight(n) is c(n) factor.
# 
# Then add arch and conductor terms.

import time

def sieve_base_primes(N):
 """Return primes up to sqrt(N)+1 for segmented sieve."""
 M = int(np.sqrt(N))+2
 s = np.ones(M, dtype=bool); s[:2]=False
 for i in range(2, int(np.sqrt(M))+1):
 if s[i]: s[i*i::i]=False
 return np.nonzero(s)[0]

def iter_prime_powers_segmented(N, segment_size=2*10**6):
 """Yield arrays (n_arr, logp_arr) where n_arr are prime powers (could be p^1 or p^k) in a chunk.
 Includes p, p^2, p^3, ... <= N.
 We yield in segments based on the underlying prime, but separately handle higher powers."""
 base_primes = sieve_base_primes(N)
 # First, yield p^k for k>=2 (relatively rare, ~sqrt(N) of them) — emit all at end as one chunk
 # Yield primes in segments
 low = 2
 while low <= N:
 high = min(low + segment_size - 1, N)
 size = high - low + 1
 seg = np.ones(size, dtype=bool)
 for p in base_primes:
 if p*p > high: break
 start = max(p*p, ((low + p - 1)//p)*p)
 seg[start - low::p] = False
 # Mark 0, 1 in segment
 if low == 2:
 seg[0] = True
 if size >= 2: seg[1] = True
 elif low <= 1:
 seg[:max(0, 2-low)] = False
 primes_seg = np.nonzero(seg)[0] + low
 primes_seg = primes_seg[primes_seg >= 2]
 if len(primes_seg) > 0:
 logp = np.log(primes_seg.astype(np.float64))
 yield primes_seg.astype(np.float64), logp
 low = high + 1
 # Now higher powers p^k, k>=2
 pp_n = []
 pp_logp = []
 for p in base_primes:
 if p*p > N: break
 pk = p*p
 while pk <= N:
 pp_n.append(pk)
 pp_logp.append(np.log(p))
 pk *= p
 if pp_n:
 yield np.array(pp_n, dtype=np.float64), np.array(pp_logp, dtype=np.float64)

# Quick test:
total = 0
for n_arr, logp in iter_prime_powers_segmented(100):
 print(n_arr.tolist(), logp[:3].tolist())
 total += len(n_arr)
print(f"Total prime powers <= 100: {total}")


[2.0, 3.0, 5.0, 7.0, 11.0, 13.0, 17.0, 19.0, 23.0, 29.0, 31.0, 37.0, 41.0, 43.0, 47.0, 53.0, 59.0, 61.0, 67.0, 71.0, 73.0, 79.0, 83.0, 89.0, 97.0] [0.6931471805599453, 1.0986122886681098, 1.6094379124341003]
[4.0, 8.0, 16.0, 32.0, 64.0, 9.0, 27.0, 81.0, 25.0, 49.0] [0.6931471805599453, 0.6931471805599453, 0.6931471805599453]
Total prime powers <= 100: 35


In [32]:
# Good, 35 prime powers <= 100, matches the spec count for primes.
# Now build the main M_arith computation:

def compute_M_arith_prime_part(T0, sigma, J, X, chi_n_func, segment_size=2*10**6, verbose=False):
 """Compute the prime-sum contribution to M_arith[m,n] as a JxJ complex matrix.
 
 Returns: P_mat = sum_{n=p^k, n<=X} chi_n_func(n) Lambda(n)/sqrt(n) * (-1) * [K_{m,n}(log n) + K_{m,n}(-log n)]
 
 K_{m,n}(x) = (1/2pi) i^{m-n} e^{i T0 x} S(m,n; x/sigma)
 S(m,n;alpha) = exp(-alpha^2/4) (i alpha / sqrt(2))^{|m-n|} sqrt(min!/max!) L_{min}^{|m-n|}(alpha^2/2)
 
 Note: chi_n_func(n_arr) returns array of complex/real coefficients c(n).
 For ζ: c(n) = 1.
 For L_chi CORRECTED: c(n) = 2 * Re(chi(n)) (which absorbs the factor-of-2)
 AND we use SYMMETRIC formula with [K(log)+K(-log)]/2.
 Equivalently, simpler: c(n) = Re(chi(n)) and formula is [K(log)+K(-log)] (no /2).
 
 For consistency, define: prime contribution =
 - sum_n c(n) Lambda(n)/sqrt(n) * 0.5 * [K(log n) + K(-log n)] (the "symmetric" form)
 
 The caller should pass c(n) appropriately:
 - ζ symmetric: c(n) = 2 (gives -sum 2*1*Lambda/sqrt(n) * 0.5*[K+K])
 actually wait — for ζ EF (over both ±gamma), arith side is "-2 sum Lambda(n)/sqrt(n) F-hat(log n)"
 which in our K notation (with K(log n)+K(-log n) being the cosine-like symmetric part) gives:
 -2 sum Lambda(n)/sqrt(n) * 0.5*(K(log n)+K(-log n)) = -sum Lambda(n)/sqrt(n) (K(log n)+K(-log n))
 So c(n) = 2 for ζ.
 - L_chi corrected (sym): c(n) = 4 * Re(chi(n)) ? Let me re-derive.

 Hmm I keep confusing factors. Let me start fresh with a SINGLE convention and derive cleanly.
 """
 pass

# Let me restart with clean derivation.
print("Restarting clean derivation")


Restarting clean derivation


In [33]:
# Clean derivation, single convention.
# 
# Test function h: R -> C (the spectral side, complex-valued allowed).
# Mellin/Fourier-like transform: h-check(x) = (1/2pi) integral h(t) e^{i t x} dt.
# 
# Weil EF for L(s, F) (general L-function with completed Lambda(s,F), conductor q, degree d, 
# archimedean parameters {kappa_j}):
# 
# sum_{rho: zero of L(s,F)} h(gamma_rho) = ... LHS
# h-check(0) * log(q/pi^d) ... (a) conductor
# + (1/2pi) integral_R h(t) sum_j Re[Psi(1/4 + kappa_j/2 + i t/2)] dt ... (b) arch (after pulling Re)
# - sum_{n>=2} Lambda(n)/sqrt(n) * [a_F(n) h-check(-log n) + conj(a_F(n)) h-check(log n)]
# ... (c) primes
# + (Polar terms if L has poles, e.g., for ζ at s=1.) ... (d)
# 
# where a_F(n) is the n-th coefficient of -L'/L (so for L(chi): a_F(n) = chi(n); for ζ: a_F(n) = 1).
# 
# Note: rho = 1/2 + i gamma_rho, gamma_rho can be complex but for GRH zeros are real.
# 
# For ζ (q=1, d=1, kappa=0, polar at s=1):
# sum_gamma h(gamma) = -h-check(0) log(pi) + (1/2pi) ∫ h(t) Re[Psi(1/4 + it/2)] dt
# - 2 sum_n Lambda(n)/sqrt(n) Re[h-check(log n)] (since a_F real)
# + h(i/2) + h(-i/2) (polar contributions)
# 
# Wait: should be h evaluated at imaginary points corresponding to s=0,1. s=1 -> rho=1, gamma=-i*(rho-1/2)... 
# Let me check convention: gamma = (rho - 1/2)/i so rho = 1/2 + i gamma. s=1 -> i gamma = 1/2 -> gamma = -i/2.
# So h(-i/2) and h(i/2) (for s=0). Sign depends on residue.
# 
# Actually for ζ: poles of ζ at s=1, ζ has no pole at s=0 (it's regular). But the completed function 
# Λ(s) = pi^{-s/2} Γ(s/2) ζ(s) has poles at s=0, 1 (from Γ(s/2) at s=0 and ζ at s=1). So the EF picks up
# residues from both, contributing h(i/2) + h(-i/2).

# For the sesquilinear form M[m,n], the test function is 
# h(t) = conj(F_m(t)) F_n(t)
# where F_n(t) = sqrt(sigma) (-i)^n phi_n(sigma(t-T0)).
# Then conj(F_m(t)) F_n(t) = sigma * i^m * (-i)^n * phi_m(sigma(t-T0)) phi_n(sigma(t-T0))
# = sigma * i^{m-n} * phi_m(.) phi_n(.)
# 
# h-check(x) = (1/2pi) integral conj(F_m)(t) F_n(t) e^{i t x} dt
# = (sigma/(2pi)) i^{m-n} integral phi_m(sigma(t-T0)) phi_n(sigma(t-T0)) e^{i t x} dt
# substituting u = sigma(t-T0), t = T0 + u/sigma, dt = du/sigma:
# = (sigma/(2pi)) i^{m-n} e^{i T0 x} (1/sigma) integral phi_m(u) phi_n(u) e^{i x u/sigma} du
# = (1/(2pi)) i^{m-n} e^{i T0 x} S(m,n; x/sigma)
# where S(m,n;alpha) = ∫ phi_m phi_n e^{i alpha u} du = (formula above).

# So h-check(x) = K_{m,n}(x) as before. Good.

# Now the Weil EF gives:
# sum_rho conj(F_m(gamma)) F_n(gamma) = (conductor) + (arch) + (primes) + (polar)
# 
# For ζ:
# M_arith[m,n] := RHS = (conductor + arch + primes + polar)
# This should equal M_zeros[m,n] := LHS for the "true" formula (which is just the EF identity).

# Now for L(s, chi) (non-self-dual), the EF as stated above is for the ZEROS OF L(s, chi) ALONE on LHS.
# So if we use the EF for L(s, chi) (LHS = sum_{rho of L(chi)} conj(F_m) F_n at gamma_rho),
# then RHS contains "chi(n) h-check(-log n) + chibar(n) h-check(log n)" for the prime sum.
# 
# To get the SYMMETRIC version (where prime sum has just Re(chi(n))), we add the EF for L(s, chibar):
# sum_{rho of L(chibar)} conj(F_m) F_n at gamma_rho = (conductor') + (arch') + (primes' = chibar K(-log) + chi K(log)) + (polar')
# 
# Adding both:
# sum_{rho of L(chi)} h(gamma) + sum_{rho of L(chibar)} h(gamma) 
# = 2*(conductor) + 2*(arch) [since same conductor q and same kappa_j (parity)]
# - sum_n Lambda(n)/sqrt(n) [(chi(n)+chibar(n)) (h-check(log n) + h-check(-log n))]
# = 2*(cond) + 2*(arch) - 2 sum_n Re(chi(n)) Lambda(n)/sqrt(n) (h-check(log n) + h-check(-log n))
# 
# The LHS is over zeros of L(chi) AND L(chibar). Zeros of L(chibar) are at gamma -> -gamma_chi.
# So LHS = sum_{gamma in {gammas of L(chi) and L(chibar)}} h(gamma)
# = sum_{gamma in {gammas of L(chi) and -gammas of L(chi)}} h(gamma)
# = sum over symmetric set {±gamma_chi}.
# 
# This matches our M_zeros (with 260 pos/neg L_chi zeros and their negatives -- 520 total terms,
# but actually it's just doubling).
# 
# WAIT — earlier we found tr_Mz = 13.54 by using BOTH Lchi_pos AND -Lchi_pos AND Lchi_neg AND -Lchi_neg
# (all 520 = 4*130 points), which matched r14. Good.
# So M_zeros (in r14 and here) sums over the SYMMETRIC zero set.

# Therefore the CORRECTED EF for L_chi (per our derivation) is:
# M_arith[m,n] = 2*(conductor[m,n]) + 2*(arch[m,n]) - 2 sum_n Re(chi(n)) Lambda(n)/sqrt(n) * (K(log n) + K(-log n))
# 
# (No polar terms — L(s,chi) and L(s,chibar) are entire for non-trivial chi.)

print("Corrected formula for L_chi M_arith:")
print(" M_arith = 2*Cond + 2*Arch - 2*sum Re(chi(n)) Lambda(n)/sqrt(n) (K(log n)+K(-log n))")


Corrected formula for L_chi M_arith:
 M_arith = 2*Cond + 2*Arch - 2*sum Re(chi(n)) Lambda(n)/sqrt(n) (K(log n)+K(-log n))


In [34]:
# Same logic for ζ but only ONE copy of conductor + arch (since ζ is self-dual):
# 
# For ζ EF (over zeros of ζ, which already form a symmetric set ±gamma):
# sum_rho h(gamma) = (cond) + (arch) - 2 sum_n Lambda(n)/sqrt(n) Re[h-check(log n)] (since a real)
# + h(i/2) + h(-i/2)
# Where Re[h-check(log n)] doesn't quite match (K(log n) + K(-log n))/2.
# h-check is complex; Re[h-check(log n)] = (h-check(log n) + conj(h-check(log n)))/2.
# But conj(h-check(log n)) = conj(K_{m,n}(log n)) — what is that?
# K_{m,n}(x) = (1/2pi) i^{m-n} e^{i T0 x} S(m,n; x/sigma)
# conj(K_{m,n}(x)) = (1/2pi) (-i)^{m-n} e^{-i T0 x} conj(S(m,n; x/sigma))
# S(m,n;alpha) = (formula with (i alpha)^diff) — for real alpha, conj(S) = (-1)^diff * S (since (i alpha)^diff -> (-i alpha)^diff = (-1)^diff (i alpha)^diff)
# So conj(K_{m,n}(x)) = (1/2pi) (-i)^{m-n} e^{-i T0 x} (-1)^{|m-n|} S(m,n;x/sigma)
# Note (-i)^{m-n} * (-1)^{|m-n|}: if m>=n diff=m-n, (-i)^{m-n}*(-1)^{m-n} = (i)^{m-n}. If m<n, diff=n-m, 
# (-i)^{m-n} = (-i)^{-(n-m)} = (i)^{n-m}; * (-1)^{n-m} = (-i)^{n-m} = (i)^{-(n-m)} = (i)^{m-n}. So always (i)^{m-n}.
# Then conj(K_{m,n}(x)) = (1/2pi) i^{m-n} e^{-i T0 x} S(m,n;x/sigma) = K_{m,n}(-x)? Let's check:
# K_{m,n}(-x) = (1/2pi) i^{m-n} e^{-i T0 x} S(m,n; -x/sigma) = (1/2pi) i^{m-n} e^{-i T0 x} (-1)^{|m-n|} S(m,n; x/sigma)
# These don't match unless (-1)^{|m-n|} = 1 (i.e., diff even).
# 
# So conj(K_{m,n}(log n)) != K_{m,n}(-log n) in general.
# 
# Hmm. Let me re-examine the ζ EF more carefully.
# 
# In Iwaniec-Kowalski (5.27) for self-dual L (a_F real), the prime sum is 
# -2 sum_n Lambda(n)/sqrt(n) h-check(log n) if h-check(log n) is treated as real
# OR more generally:
# - sum_n Lambda(n)/sqrt(n) (h-check(log n) + h-check(-log n)) without taking Re
# (since a_F(n) = conj(a_F(n)) = 1 for ζ).
# 
# That is: even for non-real h, the prime sum for ζ has [h-check(log n) + h-check(-log n)] times 1.
# 
# So for ζ: M_arith[m,n] = (cond) + (arch) - sum_n Lambda(n)/sqrt(n) [K_{m,n}(log n) + K_{m,n}(-log n)] + (polar)
# Let's check this works for trace by retesting.

# Implement and verify.

import time
from scipy.special import eval_genlaguerre

def compute_K_at_x(x, T0, sigma, J, Lag_cache=None):
 """Compute K[m,n](x) JxJ complex matrix. 
 If Lag_cache provided (3D array indexed by [diff, mn] of size (J,J)), use it; else compute.
 arg = (x/sigma)^2/2.
 """
 alpha = x/sigma
 arg = alpha**2/2
 if Lag_cache is None:
 Lag = np.zeros((J,J))
 # for each (m,n), Lag[m,n] = L_{min(m,n)}^{|m-n|}(arg)
 for diff in range(J):
 a = diff
 mn_max = J - 1 - diff
 row = np.zeros(mn_max+1)
 row[0] = 1.0
 if mn_max >= 1:
 row[1] = 1.0 + a - arg
 for nn in range(1, mn_max):
 row[nn+1] = ((2*nn+1+a-arg)*row[nn] - (nn+a)*row[nn-1])/(nn+1)
 # Place into Lag at positions (mn, mn+diff) and (mn+diff, mn)
 for mn in range(mn_max+1):
 Lag[mn, mn+diff] = row[mn]
 Lag[mn+diff, mn] = row[mn]
 else:
 Lag = Lag_cache
 
 # Build S(m,n;alpha) and K
 mn_arr = np.minimum.outer(np.arange(J), np.arange(J))
 mx_arr = np.maximum.outer(np.arange(J), np.arange(J))
 diff = mx_arr - mn_arr
 log_ratio = 0.5*(gammaln(mn_arr+1) - gammaln(mx_arr+1))
 S = np.exp(-alpha**2/4 + log_ratio) * (1j*alpha/np.sqrt(2))**diff * Lag
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 pref = np.outer(pow_m, pow_n_conj)
 K = (1.0/(2*np.pi)) * pref * np.exp(1j*T0*x) * S
 return K

# Test K(0):
K0 = compute_K_at_x(0.0, 46.13, 1.0, 5)
print(K0.diagonal())


[0.15915494+0.j 0.15915494+0.j 0.15915494+0.j 0.15915494+0.j
 0.15915494+0.j]


In [35]:
# Now write the BATCHED accumulator for prime sum contributions to M_arith.
# 
# For each prime power n, compute K_{m,n}(log n) + K_{m,n}(-log n) (JxJ matrix) and multiply by 
# -coefficient(n) Lambda(n)/sqrt(n), then accumulate.
# 
# Vectorized: process batch of N_batch primes, compute all (m,n) entries simultaneously.

def accumulate_prime_sum(T0, sigma, J, X, coeffs_func, segment_size=2*10**6, verbose=False):
 """
 Accumulate prime sum contribution to M_arith[m,n]:
 P[m,n] = - sum_{n_pp=p^k <= X} coeffs_func(n_pp) Lambda(n_pp)/sqrt(n_pp) * [K_{m,n}(log n_pp) + K_{m,n}(-log n_pp)]
 
 Returns: P (JxJ complex matrix).
 
 coeffs_func(n_array): returns array of coefficients c(n) of same length.
 """
 P = np.zeros((J, J), dtype=np.complex128)
 
 # Precompute matrix indices
 mn_arr = np.minimum.outer(np.arange(J), np.arange(J))
 mx_arr = np.maximum.outer(np.arange(J), np.arange(J))
 diff_arr = mx_arr - mn_arr
 log_ratio = 0.5*(gammaln(mn_arr+1) - gammaln(mx_arr+1))
 pref_im = np.exp(log_ratio) / np.sqrt(2)**diff_arr # absorb sqrt(2)^diff and ratio
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 K_pref = np.outer(pow_m, pow_n_conj) / (2*np.pi) # i^{m-n} / (2pi)
 sign_diff = (-1.0)**diff_arr # (-1)^|m-n|
 
 t0 = time.time()
 n_done = 0
 for n_arr_f, logp_arr_f in iter_prime_powers_segmented(X, segment_size=segment_size):
 N_batch = len(n_arr_f)
 log_n = np.log(n_arr_f)
 alpha = log_n / sigma
 arg = alpha**2 / 2
 # Compute Lag[diff, mn, batch] for diff+mn <= J-1
 Lag = np.zeros((J, J, N_batch)) # Lag[diff, mn, :] valid for mn <= J-1-diff
 for diff in range(J):
 a = diff
 mn_max = J - 1 - diff
 Lag[diff, 0] = 1.0
 if mn_max >= 1:
 Lag[diff, 1] = 1.0 + a - arg
 for nn in range(1, mn_max):
 Lag[diff, nn+1] = ((2*nn+1+a-arg)*Lag[diff, nn] - (nn+a)*Lag[diff, nn-1])/(nn+1)
 # Compute S_full[m,n,:] = exp(-alpha^2/4) * (i alpha)^|m-n| / sqrt(2)^|m-n| * sqrt(min!/max!) * Lag[diff,min,:]
 # Build S as JxJxN_batch
 S = np.zeros((J, J, N_batch), dtype=np.complex128)
 exp_factor = np.exp(-arg/2) # exp(-alpha^2/4)
 # (i alpha)^diff: factor out — compute (i alpha)^diff for each diff once
 i_alpha = 1j * alpha
 # i_alpha_pow[diff, :] = (i alpha)^diff
 i_alpha_pow = np.ones((J, N_batch), dtype=np.complex128)
 for d in range(1, J):
 i_alpha_pow[d] = i_alpha_pow[d-1] * i_alpha
 for d in range(J):
 for mn in range(J - d):
 m, n = mn, mn+d # case m<=n
 # S[m,n,:] = exp(-alpha^2/4) * (i alpha)^d / sqrt(2)^d * sqrt(mn!/(mn+d)!) * Lag[d, mn, :]
 factor = exp_factor * i_alpha_pow[d] * (pref_im[m, n]) * Lag[d, mn]
 S[m, n] = factor
 if d > 0:
 # S[n, m, :] = same expression (S is symmetric in (m,n) because phi_m phi_n product)
 S[n, m] = factor
 # Compute [K(log n) + K(-log n)] in batched form:
 # K(log n) = K_pref[m,n] * e^{i T0 log n} * S[m,n,:]
 # K(-log n) = K_pref[m,n] * e^{-i T0 log n} * (-1)^diff * S[m,n,:] (since S(m,n;-alpha)=(-1)^diff S(m,n;alpha))
 exp_pos = np.exp(1j * T0 * log_n)
 exp_neg = np.exp(-1j * T0 * log_n)
 # K_sum[m,n,:] = K_pref[m,n] * S[m,n,:] * (exp_pos + sign_diff[m,n] * exp_neg)
 sd = sign_diff[:, :, None]
 Kpr = K_pref[:, :, None]
 Ksum = Kpr * S * (exp_pos[None, None, :] + sd * exp_neg[None, None, :])
 # Multiply by -coeffs(n) * Lambda(n) / sqrt(n) and accumulate
 coeffs = coeffs_func(n_arr_f.astype(np.int64))
 weight = -coeffs * logp_arr_f / np.sqrt(n_arr_f)
 # Accumulate
 P += np.sum(Ksum * weight[None, None, :], axis=2)
 n_done += N_batch
 if verbose and n_done % (10*segment_size) < segment_size:
 print(f" Processed {n_done} prime powers, time {time.time()-t0:.1f}s")
 if verbose:
 print(f" Done: {n_done} prime powers, time {time.time()-t0:.1f}s")
 return P

# Quick test for ζ at X=1e5
def coeffs_zeta(n_arr): return np.ones_like(n_arr, dtype=np.float64)

T0=46.13; sigma=1.0; J=12
P_test = accumulate_prime_sum(T0, sigma, J, 10**5, coeffs_zeta, segment_size=2*10**5)
print("Prime sum trace (ζ, X=1e5):", np.trace(P_test).real)
print("Expected (from direct calc earlier):")
# Direct: prime_sum_dir was at X=1e7 = -0.114
log_n = np.log(ns_pp); fac_dir = (1.0/np.pi)*np.exp(-(log_n/1.0)**2/4)*np.cos(46.13*log_n)*eval_genlaguerre(11,1,(log_n/1.0)**2/2)
print(" Direct X=1e5:", -np.sum(logp_pp/np.sqrt(ns_pp)*fac_dir))


Prime sum trace (ζ, X=1e5): -0.11427788526162841
Expected (from direct calc earlier):
 Direct X=1e5: -0.11427788526162819


In [36]:
# Match! Now finish ζ M_arith and verify trace gate.

def compute_arch_diag_zeta(T0, sigma, J, npts=4001, R=None):
 """Compute the JxJ arch matrix element diagonal sum for ζ.
 Arch[m,m] = (1/2pi) integral |F_m(t)|^2 Re[Psi(1/4 + i t/2)] dt
 = (1/2pi) integral phi_m(u)^2 Re[Psi(1/4 + i(T0+u/sigma)/2)] du
 Returns just the diagonal (since we'll mainly use trace + verify lambda_min separately).
 Actually let me compute the full JxJ Arch matrix.
 Arch[m,n] = (1/2pi) integral conj(F_m)(t) F_n(t) Re[Psi(1/4 + it/2)] dt
 = (1/2pi) i^{m-n} integral sigma * phi_m(sigma(t-T0)) phi_n(sigma(t-T0)) Re[Psi(...)] dt
 = (1/2pi) i^{m-n} integral phi_m(u) phi_n(u) Re[Psi(1/4 + i (T0+u/sigma)/2)] du
 """
 if R is None: R = max(20.0, 6*np.sqrt(J))
 u = np.linspace(-R, R, npts)
 du = u[1]-u[0]
 P = phi_n_arr(J-1, u) # (J, npts)
 t_vals = T0 + u/sigma
 rePsi = np.array([float(mp.re(mp.psi(0, mp.mpc(0.25, t/2)))) for t in t_vals])
 # Compute integral phi_m(u) phi_n(u) Re_Psi(u) du
 # = P @ diag(Re_Psi) @ P.T * du
 integrand = P * rePsi[None, :] * du
 M_int = P @ integrand.T # (J,J)
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 pref = np.outer(pow_m, pow_n_conj) # i^{m-n}
 Arch = (1/(2*np.pi)) * pref * M_int
 return Arch

# Compute M_arith for ζ at T0=46.13, sigma=1, J=12, X=1e5 to verify
print("Computing arch for ζ ...")
t0=time.time()
Arch_zeta = compute_arch_diag_zeta(46.13, 1.0, 12)
print(f"Arch done, {time.time()-t0:.1f}s, tr(Arch) = {np.trace(Arch_zeta).real}")

# Conductor: log(1/pi) / (2pi) * I_J 
Cond_zeta = (np.log(1.0)-np.log(np.pi))/(2*np.pi) * np.eye(12, dtype=complex)
print(f"tr(Cond) = {np.trace(Cond_zeta).real}")

# Polar: h(i/2)+h(-i/2) where h(t)=conj(F_m(t))F_n(t)
def compute_polar_zeta(T0, sigma, J):
 """Polar contribution: h(i/2)+h(-i/2) where h(t)=conj(F_m(t))F_n(t).
 Use analytic continuation: F_n(t) = sqrt(sigma) (-i)^n phi_n(sigma(t-T0)).
 phi_n(z) = pi^{-1/4} H_n(z) exp(-z^2/2) / sqrt(2^n n!).
 """
 Polar = np.zeros((J,J), dtype=complex)
 for sgn in [1, -1]:
 z = sigma * (sgn * 0.5j - T0) # arg
 # phi_n(z), Hermite recurrence for complex z
 phivals = np.zeros(J, dtype=complex)
 phivals[0] = np.pi**(-0.25) * np.exp(-z*z/2)
 if J >= 2: phivals[1] = np.sqrt(2.0) * z * phivals[0]
 for nn in range(1, J-1):
 phivals[nn+1] = np.sqrt(2.0/(nn+1)) * z * phivals[nn] - np.sqrt(nn/(nn+1)) * phivals[nn-1]
 F = np.sqrt(sigma) * (-1j)**np.arange(J) * phivals
 # h(t) = conj(F_m) F_n, but at complex z, "conj" doesn't preserve analyticity.
 # The CORRECT analytic continuation of h(t) = conj(F_m(t)) F_n(t) (with F_m real-on-some-axis) 
 # is: h(z) := F_m_bar(z) F_n(z), where F_m_bar(z) = overline(F_m(conj(z))) — the "Schwartz reflection".
 # For F_m(t) = sqrt(sigma) (-i)^m phi_m(sigma(t-T0)), with phi_m polynomial coefficients real:
 # F_m_bar(z) = conj((-i)^m) phi_m(sigma(z-T0)) * sqrt(sigma) = (i)^m sqrt(sigma) phi_m(sigma(z-T0))
 # (since conj of the constant part, and phi_m's polynomial coefficients are real).
 # So h(z) = sqrt(sigma) (i)^m phi_m(sigma(z-T0)) * sqrt(sigma) (-i)^n phi_n(sigma(z-T0))
 # = sigma * i^{m-n} phi_m(.) phi_n(.)
 # which means h(z) for z complex is the analytic continuation of the formula.
 # 
 # So just compute Hbig[m,n] = sigma * i^{m-n} * phivals[m] * phivals[n].
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 pref = np.outer(pow_m, pow_n_conj)
 Hbig = sigma * pref * np.outer(phivals, phivals)
 Polar += Hbig
 return Polar

Polar_zeta = compute_polar_zeta(46.13, 1.0, 12)
print(f"tr(Polar) = {np.trace(Polar_zeta).real}")
print(f" imag = {np.trace(Polar_zeta).imag}")

# Prime part
def coeffs_zeta(n_arr): return np.ones_like(n_arr, dtype=np.float64)
P_prime_zeta = accumulate_prime_sum(46.13, 1.0, 12, 10**5, coeffs_zeta, segment_size=2*10**5)
print(f"tr(Prime) = {np.trace(P_prime_zeta).real}")

M_arith_zeta = Cond_zeta + Arch_zeta + P_prime_zeta + Polar_zeta
print(f"\ntr(M_arith) = {np.trace(M_arith_zeta).real}")
print(f"tr(M_zeros) = 3.690677 (reference)")
print(f"residual = {np.trace(M_arith_zeta).real - 3.690677}")


Computing arch for ζ ...


Arch done, 1.6s, tr(Arch) = 5.991004785184355
tr(Cond) = -2.186273038055438
tr(Polar) = 0.0
 imag = 0.0
tr(Prime) = -0.11427788526162841

tr(M_arith) = 3.690453861867289
tr(M_zeros) = 3.690677 (reference)
residual = -0.00022313813271113148


In [37]:
# 2e-4 residual at X=1e5 — same as before. So the polar terms are essentially 0 for ζ at T0=46.13,
# good. The 2e-4 is purely prime-cutoff error. Let's try X=1e6 to confirm:
P_prime_zeta_6 = accumulate_prime_sum(46.13, 1.0, 12, 10**6, coeffs_zeta, segment_size=2*10**5)
M_arith_zeta_6 = Cond_zeta + Arch_zeta + P_prime_zeta_6 + Polar_zeta
print(f"X=1e6: residual = {np.trace(M_arith_zeta_6).real - 3.690677}")

# X=1e7
P_prime_zeta_7 = accumulate_prime_sum(46.13, 1.0, 12, 10**7, coeffs_zeta, segment_size=10**6)
M_arith_zeta_7 = Cond_zeta + Arch_zeta + P_prime_zeta_7 + Polar_zeta
print(f"X=1e7: residual = {np.trace(M_arith_zeta_7).real - 3.690677}")


X=1e6: residual = 1.1666948562805146e-07


X=1e7: residual = -9.466809380853647e-08


In [38]:
# Same residuals as before, framework is consistent. At X=1e7 we're at 1e-7. r14 shows ζ at sigma=1 
# with X=1e9 reaches 1e-13. Let me try X=1e9 if feasible.

# Actually let me also try X=1e8 first:
t0=time.time()
P_prime_zeta_8 = accumulate_prime_sum(46.13, 1.0, 12, 10**8, coeffs_zeta, segment_size=2*10**6)
print(f"X=1e8 time: {time.time()-t0:.1f}s")
M_arith_zeta_8 = Cond_zeta + Arch_zeta + P_prime_zeta_8 + Polar_zeta
print(f"X=1e8: residual = {np.trace(M_arith_zeta_8).real - 3.690677}")


X=1e8 time: 24.3s
X=1e8: residual = -9.466989059347952e-08


In [39]:
# Residual NOT decreasing from 1e-7 to 1e-8 at X=1e8. Suggests there's a non-prime-cutoff residual 
# of ~1e-7 from the arch integral or polar terms. The r14 reference shows 1e-13 at X=1e9 — so 
# there must be something different in the r14 implementation. The arch integral quadrature 
# (4001 points) is probably the limit.

# Let me increase arch quadrature accuracy. Use higher npts or adaptive quadrature.
t0=time.time()
Arch_zeta_hi = compute_arch_diag_zeta(46.13, 1.0, 12, npts=20001, R=40)
print(f"Higher arch quad: tr_change = {(np.trace(Arch_zeta_hi)-np.trace(Arch_zeta)).real}")
M_arith_zeta_8b = Cond_zeta + Arch_zeta_hi + P_prime_zeta_8 + Polar_zeta
print(f"X=1e8 + better arch: residual = {np.trace(M_arith_zeta_8b).real - 3.690677}")
print(f"time: {time.time()-t0:.1f}s")


Higher arch quad: tr_change = -3.8617997688561445e-12
X=1e8 + better arch: residual = -9.467375283733759e-08
time: 8.2s


In [40]:
# Arch is converged to 1e-12. So 1e-7 residual is from something else.
# Wait — the reference value "3.690677" is only 7 digits. Maybe r14 stores it rounded.
# Let me get a more precise tr_Mz reference:
T0=46.13; sigma=1.0; J=12
# Use more zeros
N_zeros = 500
gammas_pos500 = np.array([float(mp.zetazero(n).imag) for n in range(1, N_zeros+1)])
gammas_neg500 = -gammas_pos500.copy()
all_g = np.concatenate([gammas_pos500, gammas_neg500])
u = sigma*(all_g - T0)
P = phi_n_arr(J-1, u)
tr_Mz_precise = sigma*(P*P).sum()
print(f"tr_Mz with 500 zeros: {tr_Mz_precise}")
print(f"vs r14 ref 3.690677")
print(f"vs M_arith X=1e8: {np.trace(M_arith_zeta_8).real}")
print(f"diff: {tr_Mz_precise - np.trace(M_arith_zeta_8).real}")


tr_Mz with 500 zeros: 3.6906769053295667
vs r14 ref 3.690677
vs M_arith X=1e8: 3.6906769053301094
diff: -5.426770144367765e-13


In [41]:
# EXCELLENT! When we compare to the actual precise tr_Mz (3.6906769053296), we get residual = 5e-13!
# The "1e-7" earlier was because we were comparing to the ROUNDED 3.690677 = 3.6906770 in r14.
# 
# So at X=1e8, our framework matches trace gate to 5e-13 for ζ! That's even better than X=1e9 in r14.
# This is great — we don't even need X=1e9 to demonstrate the trace gate.
# 
# Now let me check at the FULL J=28, T0=120 case where r14 might struggle. But let me first 
# get the corrected L_chi to work.

# Compute precise tr_Mz for L_chi at T0=46.13, J=12, sigma=1 (symmetric over zeros of L(chi) and L(chibar)):
Lchi_zeros_pos = np.array([float(s) for s in pickle.load(open('Lchi_zeros.pkl','rb'))])
Lchi_zeros_neg = np.array([float(s) for s in pickle.load(open('Lchi_zeros_neg.pkl','rb'))])
gpos = Lchi_zeros_pos
gneg = Lchi_zeros_neg

# Symmetric zero set: union of L(chi) zeros (gpos, gneg) and L(chibar) zeros (-gpos, -gneg)
all_g_sym = np.concatenate([gpos, gneg, -gpos, -gneg])
u = sigma*(all_g_sym - T0)
P = phi_n_arr(J-1, u)
tr_Mz_Lchi_sym = sigma*(P*P).sum()
print(f"L_chi sym tr_Mz at T0=46.13, J=12, sigma=1: {tr_Mz_Lchi_sym}")
print(f"r14 ref: 13.543189")


L_chi sym tr_Mz at T0=46.13, J=12, sigma=1: 13.54318943222916
r14 ref: 13.543189


In [42]:
# Match. Now build corrected M_arith for L_chi:
# M_arith[m,n] = 2*Cond[m,n] + 2*Arch[m,n] - 2*sum Re(chi(n))*Lambda(n)/sqrt(n)*(K(log n)+K(-log n))/2 
# = 2*Cond + 2*Arch - sum Re(chi(n)) Lambda(n)/sqrt(n) (K(log n)+K(-log n))*... 
# Wait — let me re-derive carefully.
# 
# EF for L(s,chi):
# sum_{rho of L(chi)} h(gamma) = (Cond_chi) + (Arch_chi) - sum Lambda(n)/sqrt(n) [chi(n) h-check(-log n) + chibar(n) h-check(log n)]
# EF for L(s,chibar): same Cond_chi, Arch_chi (since L(chibar) has same conductor, parity), but with chibar:
# sum_{rho of L(chibar)} h(gamma) = (Cond_chi) + (Arch_chi) - sum Lambda(n)/sqrt(n) [chibar(n) h-check(-log n) + chi(n) h-check(log n)]
# Add them:
# LHS_sym = sum over symmetric zero set = 2*Cond_chi + 2*Arch_chi - sum Lambda(n)/sqrt(n) [(chi+chibar) (h-check(log n)+h-check(-log n))]
# = 2*Cond_chi + 2*Arch_chi - 2 sum Re(chi(n)) Lambda(n)/sqrt(n) [h-check(log n) + h-check(-log n)]
# 
# So for the CORRECTED formula (sum over symmetric zero set):
# M_arith[m,n] = 2*Cond_chi + 2*Arch_chi - 2 sum Re(chi(n)) Lambda(n)/sqrt(n) [K_{m,n}(log n) + K_{m,n}(-log n)]
# 
# Where Cond_chi = (log q - d log pi) * K(0) with q=5, d=1: log(5/pi) * K(0) = log(5/pi) * I/(2pi).
# Arch_chi = (1/2pi) integral conj(F_m)(t) F_n(t) Re[Psi(1/4 + kappa/2 + i t/2)] dt with kappa=1 for chi odd (a=1).
# 
# Now my accumulate_prime_sum gives - sum c(n) Lambda(n)/sqrt(n) [K(log n) + K(-log n)].
# For the corrected formula, set c(n) = 2*Re(chi(n)). Then the result equals 
# -2 sum Re(chi(n)) Lambda(n)/sqrt(n) [K(log n) + K(-log n)] ✓

# Define chi(n) for chi mod 5: chi(0)=0, chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1.
chi_table = np.array([0+0j, 1+0j, 1j, -1j, -1+0j])
def Re_chi_mod5(n_arr):
 """Returns Re(chi(n)) for chi mod 5 (chi(2)=i, etc.)."""
 return np.real(chi_table[n_arr % 5])

# Test:
print(Re_chi_mod5(np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])))
# Expected: 1, 0, 0, -1, 0, 1, 0, 0, -1, 0, 1, 0, 0


[ 1. 0. -0. -1. 0. 1. 0. -0. -1. 0. 1. 0. -0.]


In [43]:
# Good. Re(chi) is nonzero only for n in {1,4} mod 5 (taking values {1, -1}).
# In fact for the CORRECTED form, only ~40% of primes contribute (those p == 1 or 4 mod 5).

# Build Arch for L_chi (chi odd, kappa=1):
def compute_arch_Lchi(T0, sigma, J, npts=4001, R=None):
 if R is None: R = max(20.0, 6*np.sqrt(J))
 u = np.linspace(-R, R, npts)
 du = u[1]-u[0]
 P = phi_n_arr(J-1, u)
 t_vals = T0 + u/sigma
 # Re Psi(1/4 + kappa/2 + i t/2) with kappa=1: argument is 3/4 + i t/2
 rePsi = np.array([float(mp.re(mp.psi(0, mp.mpc(0.75, t/2)))) for t in t_vals])
 integrand = P * rePsi[None, :] * du
 M_int = P @ integrand.T
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 pref = np.outer(pow_m, pow_n_conj)
 Arch = (1/(2*np.pi)) * pref * M_int
 return Arch

# Cond for L_chi: log(q/pi^d) * I/(2pi) = log(5/pi) * I / (2pi)
T0=46.13; sigma=1.0; J=12; X_test = 10**7

Arch_Lchi = compute_arch_Lchi(T0, sigma, J, npts=20001, R=40)
Cond_Lchi = np.log(5.0/np.pi)/(2*np.pi) * np.eye(J, dtype=complex)

# Corrected prime: c(n) = 2*Re(chi(n)) → accumulator gives -2 sum Re(chi) Lambda(n)/sqrt(n) (K(log)+K(-log))
def coeffs_Lchi_corrected(n_arr):
 return 2.0 * Re_chi_mod5(n_arr)

t0=time.time()
P_prime_Lchi_corr = accumulate_prime_sum(T0, sigma, J, X_test, coeffs_Lchi_corrected, segment_size=10**6, verbose=True)
print(f"L_chi prime sum done, time {time.time()-t0:.1f}s")

# Corrected M_arith = 2*Cond + 2*Arch + P_prime (P_prime already has -2 factor via coeffs)
M_arith_Lchi_corr = 2*Cond_Lchi + 2*Arch_Lchi + P_prime_Lchi_corr
print(f"tr(M_arith_Lchi corrected, X=1e7) = {np.trace(M_arith_Lchi_corr).real}")
print(f"tr(M_zeros_Lchi sym) = {tr_Mz_Lchi_sym}")
print(f"residual = {np.trace(M_arith_Lchi_corr).real - tr_Mz_Lchi_sym}")


 Processed 78498 prime powers, time 0.8s


 Processed 148933 prime powers, time 1.0s


 Processed 216816 prime powers, time 1.2s


 Processed 283146 prime powers, time 1.4s
 Processed 348513 prime powers, time 1.6s


 Processed 412849 prime powers, time 1.8s
 Processed 476648 prime powers, time 2.0s


 Processed 539777 prime powers, time 2.2s
 Processed 602489 prime powers, time 2.4s


 Processed 664579 prime powers, time 2.6s
 Processed 665134 prime powers, time 2.6s
 Done: 665134 prime powers, time 2.6s
L_chi prime sum done, time 2.6s
tr(M_arith_Lchi corrected, X=1e7) = 13.543189432222537
tr(M_zeros_Lchi sym) = 13.54318943222916
residual = -6.624034654123534e-12


In [44]:
# 🎉 PERFECT! Residual = 6.6e-12 at X=1e7 with corrected formula! That's well below the 1e-7 floor 
# for ζ, and ~10^8 times better than the original ~1e-4 residual.

# The hypothesis is essentially confirmed at this single point. Now we need to do the full grid 
# and compute lambda_min as well.

# Let me also compute the FULL M_arith (not just trace) and compute lambda_min.

# Compute M_zeros for L_chi (symmetric):
def build_Mzeros_sym(gpos, gneg, T0, sigma, J):
 """Symmetric M_zeros: sum over zeros of L(chi) and L(chibar) = union with negation."""
 all_g = np.concatenate([gpos, gneg, -gpos, -gneg])
 u = sigma * (all_g - T0)
 P = phi_n_arr(J-1, u)
 PP = (P @ P.T) * sigma
 powers = (1j)**np.arange(J)
 M = np.outer(powers, np.conj(powers)) * PP
 return M

t0=time.time()
Mz_Lchi = build_Mzeros_sym(gpos, gneg, T0, sigma, J)
Ma_Lchi = M_arith_Lchi_corr
Q_Lchi = Mz_Lchi - Ma_Lchi
# Symmetrize and take eigenvalues
Q_sym = (Q_Lchi + Q_Lchi.conj().T)/2
eigvals = np.linalg.eigvalsh(Q_sym)
print(f"L_chi CORRECTED at T0=46.13, J=12, X=1e7:")
print(f" tr(M_zeros) = {Mz_Lchi.trace().real:.6f}")
print(f" tr(M_arith) = {Ma_Lchi.trace().real:.6f}")
print(f" trace residual = {(Mz_Lchi-Ma_Lchi).trace().real:.6e}")
print(f" lambda_min = {eigvals.min():.6e}")
print(f" lambda_max = {eigvals.max():.6e}")
print(f"r14 original: lambda_min = -6.696798e-05, trace residual = -6.789952e-04")
print(f"time: {time.time()-t0:.1f}s")


L_chi CORRECTED at T0=46.13, J=12, X=1e7:
 tr(M_zeros) = 13.543189
 tr(M_arith) = 13.543189
 trace residual = 6.621925e-12
 lambda_min = 5.161240e-13
 lambda_max = 5.840264e-13
r14 original: lambda_min = -6.696798e-05, trace residual = -6.789952e-04
time: 0.0s


In [45]:
# 🚀 OUTSTANDING! With the corrected formula:
# - Trace residual went from -6.79e-04 to +6.62e-12 (~8 orders of magnitude improvement)
# - lambda_min went from -6.70e-05 to +5.16e-13 (NUMERICAL FLOOR, POSITIVE)
# 
# This confirms hypothesis H1!
# 
# Wait — lambda_min and lambda_max are nearly EQUAL (5.16e-13 vs 5.84e-13). That's suspicious 
# (the matrix is nearly zero). Let me check:
print("All eigenvalues:")
print(eigvals)
print()
print(f"||Q||_F = {np.linalg.norm(Q_Lchi):.4e}")
print(f"||M_zeros||_F = {np.linalg.norm(Mz_Lchi):.4e}")
print(f"||M_arith||_F = {np.linalg.norm(Ma_Lchi):.4e}")


All eigenvalues:
[5.16123952e-13 5.31957821e-13 5.37620190e-13 5.42697543e-13
 5.46026164e-13 5.47909935e-13 5.55168985e-13 5.61118527e-13
 5.64537349e-13 5.65323708e-13 5.69414649e-13 5.84026407e-13]

||Q||_F = 1.9126e-12
||M_zeros||_F = 5.6140e+00
||M_arith||_F = 5.6140e+00


In [46]:
# Q is essentially the ZERO matrix (norm 1.9e-12, vs M norms ~5.6). This means M_arith ≈ M_zeros 
# as MATRICES (not just trace). The eigenvalues are all ~5e-13 (numerical floor).
# 
# This is STRONG evidence that the corrected formula is exactly right at this point (T0=46.13, J=12).
# 
# Note: at X=1e7 with J=12 sigma=1, we're already at the numerical floor. The spec says for narrower 
# σ=1, J>20 we may need X=1e9. Let me confirm by testing larger J.

t0=time.time()
T0=46.13; sigma=1.0; J=24; X_test = 10**7

# Recompute everything for J=24
Mz_J24 = build_Mzeros_sym(gpos, gneg, T0, sigma, J)
print(f"tr(Mz J=24) = {Mz_J24.trace().real}")

Arch_Lchi_J24 = compute_arch_Lchi(T0, sigma, J, npts=40001, R=60)
Cond_Lchi_J24 = np.log(5.0/np.pi)/(2*np.pi) * np.eye(J, dtype=complex)
P_prime_Lchi_corr_J24 = accumulate_prime_sum(T0, sigma, J, X_test, coeffs_Lchi_corrected, segment_size=10**6)
Ma_Lchi_J24 = 2*Cond_Lchi_J24 + 2*Arch_Lchi_J24 + P_prime_Lchi_corr_J24
print(f"tr(Ma J=24) = {Ma_Lchi_J24.trace().real}")
print(f"trace residual = {(Mz_J24-Ma_Lchi_J24).trace().real:.4e}")

Q_J24 = Mz_J24 - Ma_Lchi_J24
Q_sym = (Q_J24 + Q_J24.conj().T)/2
eigvals_J24 = np.linalg.eigvalsh(Q_sym)
print(f" λ_min = {eigvals_J24.min():.4e}, λ_max = {eigvals_J24.max():.4e}")
print(f" ||Q||_F = {np.linalg.norm(Q_J24):.4e}")
print(f"time: {time.time()-t0:.1f}s")


tr(Mz J=24) = 27.238677424433142


tr(Ma J=24) = 27.23869794847068
trace residual = -2.0524e-05
 λ_min = -2.8762e-05, λ_max = 8.1439e-06
 ||Q||_F = 2.9893e-05
time: 25.5s


In [47]:
# At J=24 with X=1e7 we see 2e-5 residual. This is the prime-cutoff artifact mentioned in the spec.
# For the production run, we need X=1e9. Let's see if X=1e8 helps:
t0=time.time()
P_prime_Lchi_corr_J24_X8 = accumulate_prime_sum(46.13, 1.0, 24, 10**8, coeffs_Lchi_corrected, segment_size=2*10**6)
print(f"X=1e8 prime sum time: {time.time()-t0:.1f}s")
Ma_X8 = 2*Cond_Lchi_J24 + 2*Arch_Lchi_J24 + P_prime_Lchi_corr_J24_X8
trres = (Mz_J24-Ma_X8).trace().real
print(f" trace residual = {trres:.4e}")
Q_X8 = Mz_J24 - Ma_X8
Qs = (Q_X8+Q_X8.conj().T)/2
eigvals_X8 = np.linalg.eigvalsh(Qs)
print(f" λ_min = {eigvals_X8.min():.4e}, λ_max = {eigvals_X8.max():.4e}")
print(f" ||Q||_F = {np.linalg.norm(Q_X8):.4e}")


X=1e8 prime sum time: 71.4s
 trace residual = 9.9183e-12
 λ_min = -2.4564e-11, λ_max = 3.5331e-11
 ||Q||_F = 4.3032e-11


In [48]:
# X=1e8 already brings us to numerical floor (3e-11) for J=24. Massive improvement over 2e-5 at X=1e7.
# The cutoff X=1e9 is generous; X=1e8 should suffice for most points.
# 
# Time budget analysis: at X=1e8 J=24 single point: 71s. For full grid 5*7=35 points, with average 
# J=16 (time scales as ~J^2), total ~35*40s = 1400s. Manageable.
# 
# Let me also check at the most extreme J=28, T0=120 to be sure:
t0=time.time()
T0_t=120.0; sigma_t=1.0; J_t=28
Mz_t = build_Mzeros_sym(gpos, gneg, T0_t, sigma_t, J_t)
print(f"tr(Mz T0=120 J=28) = {Mz_t.trace().real}")

Arch_t = compute_arch_Lchi(T0_t, sigma_t, J_t, npts=40001, R=80)
Cond_t = np.log(5.0/np.pi)/(2*np.pi) * np.eye(J_t, dtype=complex)
P_t = accumulate_prime_sum(T0_t, sigma_t, J_t, 10**8, coeffs_Lchi_corrected, segment_size=2*10**6)
Ma_t = 2*Cond_t + 2*Arch_t + P_t
print(f"tr(Ma) = {Ma_t.trace().real}, residual = {(Mz_t-Ma_t).trace().real:.4e}")
Q_t = Mz_t - Ma_t
Qs = (Q_t + Q_t.conj().T)/2
ev = np.linalg.eigvalsh(Qs)
print(f" λ_min = {ev.min():.4e}, λ_max = {ev.max():.4e}")
print(f"time: {time.time()-t0:.1f}s")


tr(Mz T0=120 J=28) = 40.3333171007786


tr(Ma) = 40.333317128414315, residual = -2.7636e-08
 λ_min = -3.7505e-08, λ_max = 9.8495e-09
time: 107.2s


In [49]:
# T0=120, J=28 still has 3e-8 residual at X=1e8 — could improve at X=1e9. But still a HUGE improvement
# over r14's ~1e-4. Note that the larger T0 and J means more sensitivity to the high-frequency 
# oscillations in cos(T0 log p), so we need slightly more primes.
# 
# For the full r16 grid let's use X=1e9 to match the spec. Time estimate for X=1e9: 
# X=1e8 took ~107s for J=28; X=1e9 will be ~10x = ~1000s per point. That's too slow for 35 points.
#
# Compromise: use X=1e8 throughout (still vastly better than r14's situation with X=1e9 needed for ζ).
# The corrected formula already pushes residual to ~1e-11 for most points and ~1e-8 for worst case.
# Compared to r14 original ~1e-4, this is a 4-8 orders of magnitude improvement.
#
# Actually let me think: the spec said "r16" used X=1e9. To be FAITHFUL to the task, let me try to 
# run X=1e9 for the production runs. Total time for X=1e9 at full grid:
# - J=4 (5 T0's): ~30s each * 5 = 150s
# - J=8: ~40s * 5 = 200s
# - J=12: ~50s * 5 = 250s
# - J=16: ~70s * 5 = 350s
# - J=20: ~90s * 5 = 450s
# - J=24: ~120s * 5 = 600s
# - J=28: ~150s * 5 = 750s (extrapolated; actual could be more)
# Total: ~2750s. That fits in the budget.
# 
# Actually a smart approach: process the prime stream ONCE for each T0, accumulating for all J=28 
# (largest), then use submatrices for smaller J. This way the J<=28 cost is paid once.
# 
# Let me restructure: per T0, do one full pass over primes with J=28, accumulating 28x28 K matrices.
# Then for J<28, slice to JxJ submatrix.
# 
# Time: 5 T0 * (~250s for J=28 at X=1e9 prime pass) = 1250s. Plus M_zeros and Arch for all J. 
# Total ~1500s. Good.

# Wait — let me actually do X=1e8 first since it gives 1e-11 for most points, and only fall back
# to X=1e9 if needed. We can compare with r14's original. Plus the X=1e7 case for J=12 already 
# showed Q_F ~ 1e-12 (machine precision) — the prime cutoff issue mostly matters at large J.

# DECISION: Use X=1e9 for the production run as specified, but optimize by processing all 5 T0's 
# in one prime pass (since the Laguerre+S computation is T0-independent).
# Actually no: Laguerre and S DO depend on alpha = log n / sigma (T0-independent), 
# but e^{i T0 log n} depends on T0. So we can compute S once and multiply by exp(±i T0 log n) for each T0.

print("Restructuring engine for efficient multi-T0 processing")


Restructuring engine for efficient multi-T0 processing


In [50]:
def accumulate_prime_sum_multi_T0(T0_list, sigma, J, X, coeffs_func, segment_size=2*10**6, verbose=False):
 """Compute prime sum P[T0_idx, m, n] for all T0 in list simultaneously.
 P[T0,m,n] = - sum_n_pp coeffs(n_pp) Lambda(n_pp)/sqrt(n_pp) * [K_T0(log n_pp) + K_T0(-log n_pp)]
 """
 nT0 = len(T0_list)
 T0_arr = np.array(T0_list)
 P = np.zeros((nT0, J, J), dtype=np.complex128)
 
 mn_arr = np.minimum.outer(np.arange(J), np.arange(J))
 mx_arr = np.maximum.outer(np.arange(J), np.arange(J))
 diff_arr = mx_arr - mn_arr
 log_ratio = 0.5*(gammaln(mn_arr+1) - gammaln(mx_arr+1))
 pref_im = np.exp(log_ratio) / np.sqrt(2)**diff_arr
 pow_m = (1j)**np.arange(J)
 pow_n_conj = (-1j)**np.arange(J)
 K_pref = np.outer(pow_m, pow_n_conj) / (2*np.pi) # (J,J)
 sign_diff = (-1.0)**diff_arr
 
 t0_time = time.time()
 n_done = 0
 for n_arr_f, logp_arr_f in iter_prime_powers_segmented(X, segment_size=segment_size):
 N_batch = len(n_arr_f)
 log_n = np.log(n_arr_f)
 alpha = log_n / sigma
 arg = alpha**2 / 2
 
 # Lag[diff, mn, batch]
 Lag = np.zeros((J, J, N_batch))
 for diff in range(J):
 a = diff
 mn_max = J - 1 - diff
 Lag[diff, 0] = 1.0
 if mn_max >= 1:
 Lag[diff, 1] = 1.0 + a - arg
 for nn in range(1, mn_max):
 Lag[diff, nn+1] = ((2*nn+1+a-arg)*Lag[diff, nn] - (nn+a)*Lag[diff, nn-1])/(nn+1)
 
 # exp_factor[batch]
 exp_alpha = np.exp(-arg/2)
 # (i alpha)^diff
 i_alpha = 1j * alpha
 i_alpha_pow = np.ones((J, N_batch), dtype=np.complex128)
 for d in range(1, J):
 i_alpha_pow[d] = i_alpha_pow[d-1] * i_alpha
 
 # Build S[m,n,batch] for m<=n (then symmetrize)
 S = np.zeros((J, J, N_batch), dtype=np.complex128)
 for d in range(J):
 for mn in range(J - d):
 m, n_idx = mn, mn + d
 factor = exp_alpha * i_alpha_pow[d] * pref_im[m, n_idx] * Lag[d, mn]
 S[m, n_idx] = factor
 if d > 0:
 S[n_idx, m] = factor
 
 # weight[batch] = - coeffs(n) * Lambda(n) / sqrt(n)
 coeffs_n = coeffs_func(n_arr_f.astype(np.int64))
 weight = -coeffs_n * logp_arr_f / np.sqrt(n_arr_f) # (N_batch,)
 
 # For each T0: compute exp(±i T0 log n), build K_sum, accumulate
 # K_sum[T0,m,n,:] = K_pref[m,n] * S[m,n,:] * (exp_pos + sign_diff * exp_neg)
 # P[T0,m,n] += sum over batch
 # Precompute S * weight (J,J,N_batch)
 S_w = S * weight[None, None, :]
 # For each T0:
 for i_T0, T0_val in enumerate(T0_arr):
 T0_log_n = T0_val * log_n
 exp_pos = np.exp(1j * T0_log_n)
 exp_neg = np.exp(-1j * T0_log_n)
 # bracket[m,n,batch] = exp_pos + sign_diff[m,n] * exp_neg
 # P_inc[m,n] = K_pref[m,n] * sum_batch S_w[m,n,:] * bracket[m,n,:]
 # = K_pref[m,n] * (sum_batch S_w[m,n,:]*exp_pos + sign_diff[m,n]*sum_batch S_w[m,n,:]*exp_neg)
 sum_pos = np.einsum('mnk,k->mn', S_w, exp_pos)
 sum_neg = np.einsum('mnk,k->mn', S_w, exp_neg)
 P[i_T0] += K_pref * (sum_pos + sign_diff * sum_neg)
 n_done += N_batch
 if verbose and n_done % (50*segment_size) < segment_size:
 print(f" {n_done} prime powers, {time.time()-t0_time:.1f}s")
 if verbose:
 print(f" Total: {n_done} prime powers, {time.time()-t0_time:.1f}s")
 return P

# Quick verification: at T0=46.13, J=12, X=1e7, should match P_prime_Lchi_corr earlier (6.6e-12 trace residual)
t0=time.time()
P_multi = accumulate_prime_sum_multi_T0([46.13], 1.0, 12, 10**7, coeffs_Lchi_corrected, segment_size=10**6)
print(f"time {time.time()-t0:.1f}s, trace match check:")
print(f" multi-T0 prime trace: {np.trace(P_multi[0]).real}")
print(f" single-T0 prime trace: {np.trace(P_prime_Lchi_corr).real}")
print(f" diff: {np.trace(P_multi[0]).real - np.trace(P_prime_Lchi_corr).real}")


time 1.3s, trace match check:
 multi-T0 prime trace: -0.21387404674889215
 single-T0 prime trace: -0.21387404674888805
 diff: -4.107825191113079e-15


In [51]:
# Great, matches. Now let's estimate time for X=1e9, J=28, 5 T0:
t0=time.time()
T0_list = [30.0, 46.13, 60.0, 85.7, 120.0]
P_multi_test = accumulate_prime_sum_multi_T0(T0_list, 1.0, 28, 10**8, coeffs_Lchi_corrected, segment_size=2*10**6, verbose=True)
print(f"X=1e8 J=28 5 T0's: {time.time()-t0:.1f}s")


 148933 prime powers, 3.0s


 283146 prime powers, 5.7s


 412849 prime powers, 8.3s


 539777 prime powers, 10.9s


 664579 prime powers, 13.3s


...

 1624527 prime powers, 32.0s


 1741430 prime powers, 34.3s


 1857860 prime powers, 36.6s


 1973815 prime powers, 38.9s


 Total: 5762859 prime powers, 113.0s
X=1e8 J=28 5 T0's: 113.0s


In [52]:
# 113s for X=1e8, J=28, 5 T0's. For X=1e9 expect ~10x = 1130s ≈ 19 min. 
# We need to also redo for smaller J=4..24. But these can be done MUCH faster since they involve 
# smaller K matrices.
# 
# Actually, smartest: do J=28 once at X=1e9, get matrices for all 5 T0's. Then for J<28, the JxJ 
# matrix elements are SUBSETS of the J=28 ones (same Lag/S/K structure, just truncated). 
# So we can extract the leading JxJ block.
# 
# Hmm but the J appears in the Laguerre recurrence — for J=28 we compute Lag[0..27]. For J=12 we 
# need Lag[0..11]. The smaller is a subset of the larger.
# 
# So strategy: run ONE prime pass at J=28, X=1e9, get P_multi[T0, 0..27, 0..27]. Then extract 
# P[T0, :J, :J] for each smaller J. This gives all J in one pass!
# 
# But wait — the K matrix block for smaller J is NOT exactly the (J,J) corner of the J=28 result.
# Actually IT IS! K_{m,n}(x) for m,n in [0..J-1] is the same regardless of basis size. The basis
# only matters for the dimension of the matrix.
# 
# Verify by comparison:
P_J12 = accumulate_prime_sum_multi_T0([46.13], 1.0, 12, 10**6, coeffs_Lchi_corrected, segment_size=10**5)
P_J16 = accumulate_prime_sum_multi_T0([46.13], 1.0, 16, 10**6, coeffs_Lchi_corrected, segment_size=10**5)
print("Top-left 12x12 of P_J16 matches P_J12?")
print(np.max(np.abs(P_J12[0] - P_J16[0, :12, :12])))


Top-left 12x12 of P_J16 matches P_J12?
0.0


In [53]:
# Perfect, identical. So one J=28 pass at X=1e9 gives us all the prime sums for the entire grid.
# 
# Now let's do the production run:
# 1. Run one big prime pass: 5 T0's, J=28, X=1e9 (about 1130s expected)
# 2. Compute Arch and Cond for each (T0, J).
# 3. Build M_zeros for each (T0, J).
# 4. Compute Q, eigenvalues, trace residual, lambda_min.

import time

T0_list = [30.0, 46.13, 60.0, 85.7, 120.0]
J_list = [4, 8, 12, 16, 20, 24, 28]
sigma = 1.0
X = 10**9

print("Starting production prime sum: X=1e9, J=28, 5 T0's")
t_start = time.time()
P_prime_big = accumulate_prime_sum_multi_T0(T0_list, sigma, max(J_list), X, coeffs_Lchi_corrected, segment_size=2*10**6, verbose=True)
t_prime = time.time()-t_start
print(f"\nPrime sum done in {t_prime:.1f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [54]:
# The X=1e9 run is too slow. We have already established the key result for X=1e7-1e8: 
# corrected formula gives 10^7-10^8 improvement, hitting numerical floor. Let me complete the 
# analysis with X=1e8 instead, which already shows the effect clearly.
# Save what we have and do a smaller production run.
import time
import numpy as np

# Restart from clean state
T0_list = [30.0, 46.13, 60.0, 85.7, 120.0]
J_list = [4, 8, 12, 16, 20, 24, 28]
sigma = 1.0
X = 10**8 # Reduced from 1e9 for time budget

print(f"Production run with X={X}")
t_start = time.time()
P_prime_big = accumulate_prime_sum_multi_T0(T0_list, sigma, max(J_list), X, coeffs_Lchi_corrected, segment_size=2*10**6, verbose=False)
t_prime = time.time()-t_start
print(f"Prime sum done in {t_prime:.1f}s, shape {P_prime_big.shape}")


Production run with X=100000000


Prime sum done in 114.5s, shape (5, 28, 28)


In [55]:
# Great - 115s, well within budget. Now compute Arch matrices and assemble results
import time
t0 = time.time()

# Precompute Arch matrices for J=28 at each T0 (then slice for smaller J)
Arch_matrices = {}
for T0 in T0_list:
 Arch_matrices[T0] = compute_arch_Lchi(T0, sigma, 28, npts=40001, R=80)
print(f"Arch matrices done in {time.time()-t0:.1f}s")

# Cond JxJ at given J
def cond_Lchi(J):
 return np.log(5.0/np.pi)/(2*np.pi) * np.eye(J, dtype=complex)

# M_zeros for L_chi sym at each (T0, J)
import pickle
gpos = np.array([float(s) for s in pickle.load(open('Lchi_zeros.pkl','rb'))])
gneg = np.array([float(s) for s in pickle.load(open('Lchi_zeros_neg.pkl','rb'))])

# Assemble results
import pandas as pd

results = []
for i_T0, T0 in enumerate(T0_list):
 for J in J_list:
 # M_zeros
 Mz = build_Mzeros_sym(gpos, gneg, T0, sigma, J)
 # M_arith: 2*Cond + 2*Arch + P_prime (prime already has coefficient -2*Re(chi) implicitly via coeffs_func)
 Cond_J = cond_Lchi(J)
 Arch_J = Arch_matrices[T0][:J, :J]
 P_J = P_prime_big[i_T0, :J, :J]
 Ma = 2*Cond_J + 2*Arch_J + P_J
 Q = Mz - Ma
 Qs = (Q + Q.conj().T)/2
 ev = np.linalg.eigvalsh(Qs)
 tr_res = (Mz - Ma).trace().real
 lam_min = ev.min()
 results.append({
 'T0': T0, 'J': J,
 'tr_Mz': Mz.trace().real,
 'tr_Ma': Ma.trace().real,
 'trace_residual_corrected': tr_res,
 'lambda_min_corrected': lam_min,
 'lambda_max_corrected': ev.max(),
 'Q_norm_F': np.linalg.norm(Q),
 })

df_corr = pd.DataFrame(results)
print(df_corr.to_string())
print(f"\nAssembly time: {time.time()-t0:.1f}s")


Arch matrices done in 74.8s
 T0 J tr_Mz tr_Ma trace_residual_corrected lambda_min_corrected lambda_max_corrected Q_norm_F
0 30.00 4 4.434862 4.434862 -4.216405e-12 -1.075671e-12 -1.033535e-12 2.108461e-12
1 30.00 8 8.369126 8.369126 -8.421819e-12 -1.090666e-12 -1.013866e-12 2.978396e-12
2 30.00 12 12.130403 12.130403 -1.262246e-11 -1.109497e-12 -9.993204e-13 3.645643e-12
3 30.00 16 15.717532 15.717532 -1.682721e-11 -1.120274e-12 -9.825056e-13 4.209816e-12
4 30.00 20 20.360029 20.360029 -2.107392e-11 -1.144771e-12 -9.746620e-13 4.717174e-12
5 30.00 24 23.862475 23.862475 -8.110601e-11 -7.115988e-11 1.310178e-11 7.252304e-11
6 30.00 28 27.700386 27.700386 -2.147015e-08 -2.915385e-08 7.659675e-09 3.014336e-08
7 46.13 4 4.265677 4.265677 -4.885869e-12 -1.247177e-12 -1.203253e-12 2.443147e-12
8 46.13 8 9.547773 9.547773 -9.763967e-12 -1.254605e-12 -1.189360e-12 3.452581e-12
9 46.13 12 13.543189 13.543189 -1.463307e-11 -1.265420e-12 -1.162850e-12 4.225356e-12
10 46.13 16 18.191403 18.191403 

In [56]:
# Build comparison table with original r14 values
df_orig = pd.read_csv('r14_lambda_min_table.csv')
df_orig_Lchi = df_orig[df_orig.function == 'L_chi'].copy()
df_orig_Lchi = df_orig_Lchi.rename(columns={
 'lambda_min': 'lambda_min_original',
 'trace_residual': 'trace_residual_original',
 'tr_Mz': 'tr_Mz_original',
 'tr_Ma': 'tr_Ma_original',
}).drop(columns=['function'])

# Merge
df_compare = df_corr.merge(df_orig_Lchi[['T0','J','lambda_min_original','trace_residual_original']], on=['T0','J'])
df_compare = df_compare[['T0','J','tr_Mz','lambda_min_original','trace_residual_original',
 'lambda_min_corrected','trace_residual_corrected','Q_norm_F']]
print(df_compare.to_string())

# Save comparison
df_compare.to_csv('r17_lambda_min_corrected_comparison.csv', index=False)
print("\nSaved r17_lambda_min_corrected_comparison.csv")


 T0 J tr_Mz lambda_min_original trace_residual_original lambda_min_corrected trace_residual_corrected Q_norm_F
0 30.00 4 4.434862 -0.000149 -0.000534 -1.075671e-12 -4.216405e-12 2.108461e-12
1 30.00 8 8.369126 -0.000163 -0.001076 -1.090666e-12 -8.421819e-12 2.978396e-12
2 30.00 12 12.130403 -0.000176 -0.001625 -1.109497e-12 -1.262246e-11 3.645643e-12
3 30.00 16 15.717532 -0.000187 -0.002181 -1.120274e-12 -1.682721e-11 4.209816e-12
4 30.00 20 20.360029 -0.000198 -0.002746 -1.144771e-12 -2.107392e-11 4.717174e-12
5 30.00 24 23.862475 -0.000208 -0.003318 -7.115988e-11 -8.110601e-11 7.252304e-11
6 30.00 28 27.700386 -0.000219 -0.003898 -2.915385e-08 -2.147015e-08 3.014336e-08
7 46.13 4 4.265677 -0.000060 -0.000225 -1.247177e-12 -4.885869e-12 2.443147e-12
8 46.13 8 9.547773 -0.000064 -0.000451 -1.254605e-12 -9.763967e-12 3.452581e-12
9 46.13 12 13.543189 -0.000067 -0.000679 -1.265420e-12 -1.463307e-11 4.225356e-12
10 46.13 16 18.191403 -0.000070 -0.000908 -1.279388e-12 -1.950917e-11 4.87998

In [57]:
# RESULTS SUMMARY
# 
# For J ≤ 20 (all T0): trace_residual drops from ~1e-4 to ~1e-11 (7 orders of magnitude improvement)
# lambda_min drops from ~1e-4 to ~1e-12 (8 orders of magnitude improvement)
# For J = 24: residual ~ 1e-10, lambda_min ~ 1e-11 (still huge improvement; prime cutoff X=1e8 marginal) 
# For J = 28: residual ~ 2e-8, lambda_min ~ 1e-8 (prime cutoff X=1e8 insufficient; would need X=1e9)
# 
# This is consistent with the spec's note that X=1e9 is needed for σ=1, large J.
# 
# HYPOTHESIS CONFIRMED: The corrected formula (using Re(chi(n)) in symmetric form) eliminates 
# the systematic trace residual and brings lambda_min to the numerical floor for J ≤ 20-24.
# The residual at J=28 is a separate prime-cutoff artifact (same nature as observed for ζ in r14 
# at X=1e5), not a formula error. It would be resolved by X=1e9.

# Now let me make the final figure
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(9, 10))

# Panel A: |lambda_min| vs J for each T0, both original and corrected
ax = axes[0]
colors = {30:'C0', 46.13:'C1', 60:'C2', 85.7:'C3', 120:'C4'}
for T0 in T0_list:
 sub = df_compare[df_compare.T0 == T0]
 ax.plot(sub.J, np.abs(sub.lambda_min_original), 'o--', color=colors[T0], 
 alpha=0.5, label=f'T₀={T0} (original)')
 ax.plot(sub.J, np.abs(sub.lambda_min_corrected), 'o-', color=colors[T0], 
 label=f'T₀={T0} (corrected)')
ax.set_yscale('log')
ax.set_xlabel('Basis dimension J')
ax.set_ylabel('|λ_min|')
ax.set_title('A. |λ_min| vs J for L(χ₄ mod 5): original vs corrected formula\n(σ=1, X=10⁸)')
ax.axhline(1e-12, color='gray', ls=':', label='Numerical floor (10⁻¹²)')
ax.legend(ncol=2, fontsize=7, loc='best')
ax.grid(True, alpha=0.3)

# Panel B: |trace residual| vs J
ax = axes[1]
for T0 in T0_list:
 sub = df_compare[df_compare.T0 == T0]
 ax.plot(sub.J, np.abs(sub.trace_residual_original), 'o--', color=colors[T0], 
 alpha=0.5, label=f'T₀={T0} (original)')
 ax.plot(sub.J, np.abs(sub.trace_residual_corrected), 'o-', color=colors[T0], 
 label=f'T₀={T0} (corrected)')
ax.set_yscale('log')
ax.set_xlabel('Basis dimension J')
ax.set_ylabel('|trace residual|')
ax.set_title('B. |trace residual| vs J: corrected formula resolves systematic ~10⁻⁴ error')
ax.axhline(1e-12, color='gray', ls=':', label='Numerical floor (10⁻¹²)')
ax.legend(ncol=2, fontsize=7, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('r17_corrected_formula_convergence.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved figure: r17_corrected_formula_convergence.png")


Saved figure: r17_corrected_formula_convergence.png


In [58]:
# Save final artifact
df_compare.to_csv('r17_lambda_min_corrected_comparison.csv', index=False)
print("Files in workspace:")
import os
for f in os.listdir('.'):
 if not f.startswith('.') and f != 'memory':
 sz = os.path.getsize(f)
 print(f" {f}: {sz} bytes")


Files in workspace:
 Lchi_zeros.pkl: 7020 bytes
 r17_corrected_formula_convergence.png: 206106 bytes
 engine-spec.md: 5530 bytes
 r17_lambda_min_corrected_comparison.csv: 4950 bytes
 r14_lambda_min_table.csv: 6797 bytes
 Lchi_zeros_neg.pkl: 7148 bytes
